# Lab 3 — Validación, gobierno y proyecto final · Financiera Andes
## Modelador de Riesgo de Crédito · Academia Bayes — **Entrega 3** (50% del curso)

**Publicado el miércoles 16 de septiembre · fecha límite: domingo 27 de septiembre, 23:59 (hora de Chile).**

**Cómo se entrega:** por el canal del aula, el **link a este notebook en Colab** (con permiso de
lectura) **ejecutado de arriba a abajo sin errores**, más los **tres documentos del proyecto**
(informe técnico, model card y resumen ejecutivo) en PDF o MD.
**Atrasos:** −0,5 por día, máximo 3 días. **Notas y feedback: viernes 2 de octubre.**

Este es el **laboratorio de la semana 3** — el último, y el que define el certificado.

| | Parte | Qué construyes | Puntos |
|---|---|---|---|
| **A** | Clase 5 · Validación y monitoreo | Discriminación, estabilidad, backtesting y tu tablero | 12 |
| **B** | Clase 6 · Implementación y gobierno | **A elección, una de dos:** opción 1 con `nikodym` (B1–B5: la corrida completa, mecánica escrita) u opción 2 a mano (tareas 7–12: artefacto, contrato, trail, model card, gatillos y resumen) | 15 |

**Total del notebook: 27 puntos.** La nota de la E3 combina este notebook con los **tres
documentos del proyecto** — la pauta completa y las tres plantillas van al final de este mismo
notebook (anexos 1 y 2): no necesitas ningún otro archivo.

> 🔀 **La parte B se hace UNA sola vez, por la opción que prefieras.** Las dos valen el mismo máximo
> (15 puntos) y se corrigen con la misma pauta. Con `nikodym` la mecánica viene escrita y tú pones el
> criterio; a mano construyes tú las piezas. No hagas las dos: si entregas ambas, se corrige la de
> `nikodym`.

**La parte A** es la de la clase 5; **la parte B**, la de la clase 6, por la opción que elijas. Los
esqueletos sin completar corren solos y sus verificaciones muestran ⏭️ en vez de fallar:
«Reiniciar y ejecutar todo» nunca se bloquea por un ítem pendiente (la opción de parte B que no
hagas queda en ⏭️ y no se corrige).

> 🧩 **Este notebook es autocontenido.** No necesitas haber hecho los Labs 1 y 2: las celdas de
> referencia construyen tu muestra, tu matriz, tu modelo y tu calibración con la receta del curso, y
> con eso puedes hacer toda la Entrega 3 desde aquí (solo declara en la lectura 6b que usaste la
> lista de variables de referencia). Si sí los hiciste, puedes poner tus propias `ELEGIDAS` y tu
> cutoff donde el notebook lo indica.

**Invalidantes de la E3** (los mismos del anexo 1 §5; llevan la nota a un **máximo de
4,0**): números en los documentos que no salen del notebook · métricas solo en DEV · artefacto
que no reproduce el score y se entrega igual sin declararlo · números de otra semilla ·
modelo ajustado sobre una matriz que no es la propia.

> ⚠️ **Ojo con las fechas**: la entrega es el **domingo 27 de septiembre** y las **Fiestas Patrias
> (18 y 19) caen DENTRO del plazo**, no después. Esta es la **fecha máxima**: el plazo ya se corrió
> desde el miércoles 23 y no se corre más. Presupuesta el tiempo contando los feriados, no el
> calendario completo.

> ⚠️ **Andes no es Austral.** No esperes que tus semáforos calquen los de la demo de la clase 5: **la
> nota no premia el tablero verde, premia el diagnóstico proporcionado a lo que TUS datos muestren**
> (patrón, acción con gatillo, qué NO hacer y quién firma). Las justificaciones 5b y 6b son las que más
> pesan de la parte A. Los umbrales del curso están declarados en cada ítem; si propones otros, decláralos.

---
## 0. Identificación, muestra personal y tu modelo calibrado del Lab 2 · parte B

⚠️ Usa el **mismo correo de los Labs 1 y 2**. Tu muestra personal — y por lo
tanto todos tus números — dependen de él.

In [ ]:
ID_ALUMNO = "tu.correo@ejemplo.cl"   # ← EDITA con tu correo de inscripción

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (9, 4.2)
plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = 0.3

assert ID_ALUMNO != "tu.correo@ejemplo.cl", "⚠️ Edita ID_ALUMNO con tu correo real"
SEMILLA_PERSONAL = int(hashlib.sha256(ID_ALUMNO.strip().lower().encode()).hexdigest()[:8], 16) % 10**6
print(f"Tu semilla personal: {SEMILLA_PERSONAL}")

In [ ]:
BASE = "https://raw.githubusercontent.com/nexolabs-gh/datos-riesgo-credito/main"
PREFIJO = "andes"

clientes       = pd.read_parquet(f"{BASE}/{PREFIJO}_clientes.parquet")
solicitudes    = pd.read_parquet(f"{BASE}/{PREFIJO}_solicitudes.parquet")
comportamiento = pd.read_parquet(f"{BASE}/{PREFIJO}_comportamiento.parquet")
bureau         = pd.read_parquet(f"{BASE}/{PREFIJO}_bureau.parquet")

# ---- Muestra personal (NO modificar esta celda) ----
# Ojo: el generador AVANZA con cada llamada — si re-ejecutas solo la línea de
# mis_clientes, la muestra cambia. Re-ejecuta esta celda completa (o «Reiniciar
# y ejecutar todo») y tu muestra personal será SIEMPRE la misma.
rng_p = np.random.default_rng(SEMILLA_PERSONAL)
ids = np.sort(clientes["id_cliente"].unique())
mis_clientes = set(rng_p.choice(ids, size=int(len(ids) * 0.7), replace=False))
clientes       = clientes[clientes["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
solicitudes    = solicitudes[solicitudes["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
comportamiento = comportamiento[comportamiento["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
bureau         = bureau[bureau["id_cliente"].isin(mis_clientes)].reset_index(drop=True)
print(f"Tu muestra: {len(clientes):,} clientes · {len(solicitudes):,} solicitudes")

### Recordatorio: el diccionario de datos, a mano

Seguimos con las mismas cuatro tablas de Financiera Andes, así que dejamos el
diccionario cargado otra vez. Cada variable que fabriquemos hoy nace de una
columna de estas tablas: si no sabes qué mide la columna, no puedes defender la
variable frente a un comité.

[`diccionario_andes.md`](https://github.com/nexolabs-gh/datos-riesgo-credito/blob/main/diccionario_andes.md) ·
la celda siguiente lo muestra completo y deja `mostrar_diccionario(...)` definida.

In [ ]:
import urllib.request

from IPython.display import Markdown, display

URL_DICCIONARIO = f"{BASE}/diccionario_{PREFIJO}.md"
_DICCIONARIO = urllib.request.urlopen(URL_DICCIONARIO).read().decode("utf-8")


def mostrar_diccionario(tabla=None):
    """Muestra el diccionario de datos oficial; sin argumento, lo muestra completo."""
    if tabla is None:
        display(Markdown(_DICCIONARIO))
        return
    secciones = _DICCIONARIO.split("\n## ")[1:]
    elegidas = [s for s in secciones if s.lower().startswith(tabla.lower())]
    if not elegidas:
        disponibles = [s.split("(")[0].strip() for s in secciones]
        raise ValueError(f"No existe la tabla {tabla!r}. Disponibles: {disponibles}")
    display(Markdown("## " + elegidas[0]))


mostrar_diccionario()

### Cómo se lee este notebook

Cada ítem trae su **mecánica** ya escrita (celdas «NO modificar», que corren solas sobre tu
muestra) y una celda **«👇 TU LECTURA»** o **«👇 TU CÓDIGO — insumos»** donde va lo tuyo. Un texto
que empieza con «ESCRIBE AQUÍ» cuenta como no hecho: el ✅ de cada ítem aparece solo cuando lo
completaste, y lo que dice lo lee el corrector con la pauta.

In [ ]:
PLACEHOLDER = "ESCRIBE AQUÍ"


def completo(*insumos, minimo=60):
    """True si ningún insumo es `...`, ni un placeholder, ni un texto demasiado corto."""
    for insumo in insumos:
        if insumo is ...:
            return False
        if isinstance(insumo, str):
            if insumo.strip().startswith(PLACEHOLDER) or len(insumo.strip()) < minimo:
                return False
        elif isinstance(insumo, (list, tuple, dict)):
            if len(insumo) == 0 or any(v is ... for v in (insumo.values() if isinstance(insumo, dict) else insumo)):
                return False
    return True

### Tu Lab 1 y tu Lab 2, en celdas de referencia

Las **cinco celdas siguientes** reconstruyen tu cadena completa: matriz
(Lab 1 · parte B), herramientas de binning/WoE, modelo con scorecard
(Lab 2 · parte A) y **calibración a la TC con master scale (Lab 2 · parte B)**.
Son las mismas celdas de referencia de siempre.

> ✏️ **Si tu Lab 2 usó otras variables o los tuyos difieren** (¡lo esperable!):
> reemplaza `ELEGIDAS` por tu lista final. La precargada es la de la solución de
> referencia y existe como red de seguridad — si la usas tal cual, **decláralo
> en la justificación 6b**, como en los labs anteriores.

In [ ]:
# ---- Referencia Lab 1 · parte A: target, población y muestras ----
mora_w = comportamiento.pivot_table(index="id_cliente", columns="mes",
                                    values="dias_mora", aggfunc="first")
meses = list(mora_w.columns)
pos_mes = {m: k for k, m in enumerate(meses)}
M = mora_w.to_numpy()
fila_de = {c: k for k, c in enumerate(mora_w.index)}

cursadas = solicitudes[solicitudes["aprobada"]].copy()
cursadas["_t"] = cursadas["fecha_solicitud"].map(pos_mes)
cursadas["_i"] = cursadas["id_cliente"].map(fila_de)
peor = np.full(len(cursadas), np.nan)
for k, (i, t0) in enumerate(zip(cursadas["_i"], cursadas["_t"])):
    if t0 + 12 <= len(meses) - 1:
        peor[k] = M[i, t0 + 1: t0 + 13].max()
cursadas["peor_dpd_12m"] = peor
cursadas["malo"] = np.where(np.isnan(peor), np.nan, (peor >= 90).astype(float))
cursadas["indeterminado"] = (peor >= 30) & (peor < 90)

antig = clientes.set_index("id_cliente")["fecha_alta_cliente"].map(
    lambda s: (pd.Period(s, freq="M") - pd.Period(meses[0], freq="M")).n)
periodo = cursadas[cursadas["fecha_solicitud"].between("2024-07", "2025-06")]
periodo = periodo.sort_values("fecha_solicitud", kind="stable").drop_duplicates(
    "id_cliente", keep="first").copy()
periodo["antiguedad_meses"] = periodo["_t"] - periodo["id_cliente"].map(antig)
base = periodo[periodo["malo"].notna() & ~periodo["marca_fraude"]
               & (periodo["antiguedad_meses"] >= 6)].copy()
rng = np.random.default_rng(SEMILLA_PERSONAL)
base["muestra"] = np.where(base["fecha_solicitud"] <= "2025-02",
                           np.where(rng.random(len(base)) < 0.7, "DEV", "HO"), "OOT")
ttd_pob = cursadas[cursadas["fecha_solicitud"] >= "2025-07"].copy()
ttd_pob["antiguedad_meses"] = ttd_pob["_t"] - ttd_pob["id_cliente"].map(antig)
ttd_pob = ttd_pob[ttd_pob["antiguedad_meses"] >= 6]
ttd_pob["muestra"] = "TTD"
poblacion = pd.concat([base, ttd_pob], ignore_index=True)
print(poblacion["muestra"].value_counts().reindex(["DEV", "HO", "OOT", "TTD"]).to_string())

In [ ]:
# ---- Referencia Lab 1 · parte B: panel, fábrica, fotos y ratios ----
def pivotar(df, columna):
    ancho = df.pivot_table(index="id_cliente", columns="mes", values=columna, aggfunc="first")
    return ancho.reindex(index=mora_w.index, columns=meses).to_numpy()

PANEL = {c: pivotar(comportamiento, c) for c in
         ["dias_mora", "saldo_linea", "cupo_linea", "saldo_tc", "cupo_tc", "saldo_consumo",
          "monto_pagado", "monto_facturado", "abonos_cuenta", "saldo_ahorro", "n_productos"]}
PANEL.update({c: pivotar(bureau, c) for c in
              ["deuda_otras_inst", "n_otras_inst", "consultas_mes", "peor_mora_sistema"]})
with np.errstate(divide="ignore", invalid="ignore"):
    PANEL["uso_linea"] = np.where(PANEL["cupo_linea"] > 0,
                                  PANEL["saldo_linea"] / PANEL["cupo_linea"], np.nan)
    PANEL["uso_tc"] = np.where(PANEL["cupo_tc"] > 0,
                               PANEL["saldo_tc"] / PANEL["cupo_tc"], np.nan)
PANEL["deuda_interna"] = PANEL["saldo_linea"] + PANEL["saldo_tc"] + PANEL["saldo_consumo"]
PANEL["deuda_total"] = PANEL["deuda_interna"] + PANEL["deuda_otras_inst"]
PANEL["en_mora"] = np.where(np.isnan(PANEL["dias_mora"]), np.nan,
                            (PANEL["dias_mora"] > 0).astype(float))
PANEL["en_mora_sistema"] = np.where(np.isnan(PANEL["peor_mora_sistema"]), np.nan,
                                    (PANEL["peor_mora_sistema"] > 0).astype(float))

def prom(V):  return np.nanmean(V, axis=1)
def vmax(V):  return np.nanmax(V, axis=1)
def suma(V):  return np.nansum(V, axis=1)
def delta(V): return V[:, -1] / np.maximum(V[:, 0], 1) - 1
def recencia(V):
    evento = np.nan_to_num(V, nan=0.0) > 0
    k = evento.shape[1]
    ultimo = k - 1 - np.argmax(evento[:, ::-1], axis=1)
    return np.where(evento.any(axis=1), k - ultimo, k + 1)
AGREGADORES = {"prom": prom, "max": vmax, "suma": suma, "delta": delta, "recencia": recencia}

def agregar_ventana(matriz, filas, t0s, k, funcion):
    salida = np.full(len(filas), np.nan)
    for t0 in np.unique(t0s):
        sel = t0s == t0
        salida[sel] = funcion(matriz[filas[sel], max(t0 - k, 0):t0])   # t0 NO entra
    return salida

FABRICA = [
    ("dias_mora", "max", (3, 6, 12), "dias_mora_max_{k}m"),
    ("dias_mora", "prom", (3, 6, 12), "dias_mora_prom_{k}m"),
    ("en_mora", "suma", (3, 6, 12), "n_meses_mora_{k}m"),
    ("en_mora", "recencia", (12,), "meses_desde_mora_12m"),
    ("uso_linea", "prom", (3, 6, 12), "uso_linea_prom_{k}m"),
    ("uso_linea", "max", (3, 6, 12), "uso_linea_max_{k}m"),
    ("uso_tc", "prom", (3, 6, 12), "uso_tc_prom_{k}m"),
    ("uso_tc", "max", (3, 6, 12), "uso_tc_max_{k}m"),
    ("saldo_consumo", "prom", (3, 6, 12), "saldo_consumo_prom_{k}m"),
    ("deuda_interna", "prom", (3, 6, 12), "deuda_interna_prom_{k}m"),
    ("deuda_total", "prom", (3, 6, 12), "deuda_total_prom_{k}m"),
    ("deuda_total", "delta", (6, 12), "deuda_total_delta_{k}m"),
    ("monto_pagado", "suma", (3, 6, 12), "pagos_{k}m"),
    ("monto_facturado", "suma", (3, 6, 12), "facturacion_{k}m"),
    ("abonos_cuenta", "prom", (3, 6, 12), "abonos_prom_{k}m"),
    ("saldo_ahorro", "prom", (3, 6, 12), "saldo_ahorro_prom_{k}m"),
    ("deuda_otras_inst", "prom", (3, 6, 12), "deuda_otras_prom_{k}m"),
    ("deuda_otras_inst", "delta", (6,), "delta_deuda_otras_{k}m"),
    ("n_otras_inst", "prom", (3, 6, 12), "n_otras_inst_prom_{k}m"),
    ("peor_mora_sistema", "max", (3, 6, 12), "peor_mora_sistema_{k}m"),
    ("consultas_mes", "suma", (3, 6, 12), "consultas_{k}m"),
    ("consultas_mes", "recencia", (12,), "meses_desde_consulta_12m"),
]
META = ["id_solicitud", "id_cliente", "fecha_solicitud", "_t", "_i",
        "muestra", "malo", "indeterminado", "peor_dpd_12m"]
filas_p, t0s_p = poblacion["_i"].to_numpy(), poblacion["_t"].to_numpy()
X = poblacion[META + ["antiguedad_meses", "monto_solicitado", "plazo_meses",
                      "destino", "canal"]].copy()
for serie, agg, ventanas, plantilla in FABRICA:
    for k in ventanas:
        X[plantilla.format(k=k)] = agregar_ventana(PANEL[serie], filas_p, t0s_p, k,
                                                   AGREGADORES[agg])
t_ult = np.maximum(t0s_p - 1, 0)
for serie in ["dias_mora", "saldo_linea", "cupo_linea", "saldo_tc", "cupo_tc",
              "saldo_consumo", "saldo_ahorro", "deuda_otras_inst", "n_productos",
              "peor_mora_sistema", "uso_linea", "uso_tc"]:
    X[f"{serie}_ult"] = PANEL[serie][filas_p, t_ult]
ingreso = X["abonos_prom_6m"].to_numpy()
deuda_ult = (X["saldo_linea_ult"] + X["saldo_tc_ult"]
             + X["saldo_consumo_ult"] + X["deuda_otras_inst_ult"])
X["carga_financiera"] = deuda_ult / np.maximum(ingreso, 1)
X["pago_sobre_fact_6m"] = X["pagos_6m"] / np.maximum(X["facturacion_6m"], 1)
X["pago_sobre_fact_12m"] = X["pagos_12m"] / np.maximum(X["facturacion_12m"], 1)
X["deuda_int_sobre_sistema"] = (X["saldo_linea_ult"] + X["saldo_tc_ult"]
                                + X["saldo_consumo_ult"]) / np.maximum(deuda_ult, 1)
X["ahorro_sobre_ingreso"] = X["saldo_ahorro_ult"] / np.maximum(ingreso, 1)
X["monto_sobre_ingreso"] = X["monto_solicitado"] / np.maximum(ingreso, 1)
X["cuota_est_sobre_ingreso"] = (X["monto_solicitado"] / X["plazo_meses"]) / np.maximum(ingreso, 1)
X = X.merge(clientes[["id_cliente", "edad", "sexo", "region", "estado_civil",
                      "nivel_educacional", "tipo_empleo", "renta_liquida",
                      "n_dependientes"]], on="id_cliente", how="left")
X["renta_vs_abonos"] = X["renta_liquida"] / np.maximum(ingreso, 1)

predictoras = [c for c in X.columns if c not in META]
filtro = X["malo"].notna() & ~X["indeterminado"]
dev = X[(X["muestra"] == "DEV") & filtro]
ho  = X[(X["muestra"] == "HO") & filtro]
oot = X[(X["muestra"] == "OOT") & filtro]
ttd = X[X["muestra"] == "TTD"]
print(f"TU MATRIZ: {len(X):,} filas × {len(predictoras)} candidatas")
print(f"DEV={len(dev):,} ({dev['malo'].mean():.1%} malos) · HO={len(ho):,} · "
      f"OOT={len(oot):,} · TTD={len(ttd):,}")

In [ ]:
# ---- Herramientas del curso: binning con `ref` + tabla WoE ----
def binear(x, bins=5, umbral_moda=0.35, ref=None):
    """Binning del curso. Con `ref`, los cortes se calculan en `ref` y se aplican a `x`."""
    x = pd.Series(x).reset_index(drop=True)
    base_b = x if ref is None else pd.Series(ref).reset_index(drop=True)
    if base_b.dtype == object or isinstance(base_b.dtype, pd.CategoricalDtype):
        return x.astype(str).where(x.notna(), "MISSING"), None
    if base_b.nunique(dropna=True) <= bins:
        return x.astype(str).where(x.notna(), "MISSING"), None
    moda = base_b.mode().iloc[0]
    frac_moda = (base_b == moda).mean()
    resto = base_b[base_b != moda] if frac_moda > umbral_moda else base_b
    cortes = np.unique(np.nanquantile(resto.dropna(), np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return x.astype(str).where(x.notna(), "MISSING"), None
    cortes[0], cortes[-1] = -np.inf, np.inf
    categorias = pd.cut(x, cortes)
    orden = [str(c) for c in categorias.cat.categories]
    etiquetas = categorias.astype(str)
    if frac_moda > umbral_moda:
        etiquetas = etiquetas.where(x != moda, f"= {moda:.4g}")
        orden = [f"= {moda:.4g}"] + orden
    return etiquetas.where(x.notna(), "MISSING"), orden + ["MISSING"]


def tabla_woe(x, y, bins=5, ref=None):
    """Tabla WoE de una variable y su IV (convención: target 1 = malo)."""
    etiquetas, orden = binear(x, bins, ref=ref)
    tab = pd.DataFrame({"bin": etiquetas.values, "y": pd.Series(y).values}) \
            .groupby("bin")["y"].agg(["count", "sum"])
    tab.columns = ["n", "malos"]
    if orden:
        tab = tab.reindex([o for o in orden if o in tab.index])
    tab["buenos"] = tab["n"] - tab["malos"]
    tab["tasa_malos"] = tab["malos"] / tab["n"]
    p_malos  = (tab["malos"] + 0.5) / (tab["malos"].sum() + 0.5 * len(tab))
    p_buenos = (tab["buenos"] + 0.5) / (tab["buenos"].sum() + 0.5 * len(tab))
    tab["woe"] = np.log(p_buenos / p_malos)
    tab["iv_aporte"] = (p_buenos - p_malos) * tab["woe"]
    return tab, float(tab["iv_aporte"].sum())

In [ ]:
# ---- Tu modelo del Lab 2 · parte A (red de seguridad de referencia) ----
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve

# 👇 Reemplaza por TUS variables finales del Lab 2 · parte A si difieren (lo esperable).
ELEGIDAS = ["uso_linea_prom_12m", "meses_desde_mora_12m", "uso_tc_max_12m",
            "peor_mora_sistema_12m", "monto_sobre_ingreso", "deuda_total_delta_12m",
            "consultas_12m", "uso_linea_prom_3m"]

MAPAS = {v: tabla_woe(dev[v], dev["malo"])[0]["woe"] for v in ELEGIDAS}

def a_woe(df, variables):
    """Transforma `df` al WoE de DEV: binea con cortes de DEV y mapea su WoE."""
    F = pd.DataFrame(index=range(len(df)))
    for v in variables:
        etiquetas, _ = binear(df[v], ref=dev[v])
        F[v] = etiquetas.map(MAPAS[v]).fillna(0.0).values
    return F

modelo = sm.Logit(dev["malo"].values,
                  sm.add_constant(a_woe(dev, ELEGIDAS), has_constant="add")
                  ).fit(disp=0, maxiter=200)
assert (modelo.params.drop("const") < 0).all(), \
    "⚠️ Coeficientes positivos: si son TUS variables, vuelve al Lab 2 · parte A antes de seguir"

PDO, SCORE_BASE, ODDS_BASE = 20, 600, 50
factor = PDO / np.log(2)
offset = SCORE_BASE - factor * np.log(ODDS_BASE)

def pd_de(df):
    """PD cruda del modelo (WoE y cortes de DEV, como producción)."""
    return np.asarray(modelo.predict(sm.add_constant(a_woe(df, ELEGIDAS),
                                                     has_constant="add")))

def score_desde_pd(p):
    """Score PDO desde una PD, con clip bilateral (evita ±inf en los bordes)."""
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    return offset + factor * np.log((1 - p) / p)

print(f"modelo del Lab 2 · parte A: {len(ELEGIDAS)} variables · "
      f"Gini DEV {2 * roc_auc_score(dev['malo'], pd_de(dev)) - 1:.3f}")

In [ ]:
# ---- Tu calibración del Lab 2 · parte B (red de seguridad de referencia) ----
from scipy.optimize import brentq
from scipy.special import expit, logit

MUESTRAS = {"DEV": dev, "HO": ho, "OOT": oot, "TTD": ttd}
PD_RAW = {m: pd_de(df) for m, df in MUESTRAS.items()}

gb = X[X["malo"].notna() & ~X["indeterminado"]]
TC = float(gb.groupby("fecha_solicitud")["malo"].mean().mean())   # promedio simple

lp_dev = logit(np.clip(PD_RAW["DEV"], 1e-12, 1 - 1e-12))
delta = float(brentq(lambda d: expit(lp_dev + d).mean() - TC, -5, 5, xtol=1e-12))

PD_CAL, SCORE_CAL = {}, {}
for m in MUESTRAS:
    lp = logit(np.clip(PD_RAW[m], 1e-12, 1 - 1e-12))
    PD_CAL[m] = expit(lp + delta)
    SCORE_CAL[m] = score_desde_pd(PD_CAL[m])
assert abs(PD_CAL["DEV"].mean() - TC) < 1e-9

CORTES_BANDA = [540, 560, 580, 600, 620, 640, 660]
ETIQUETAS = ["E", "D", "C2", "C1", "B2", "B1", "A2", "A1"]      # peor → mejor

def banda_de(score):
    # right=False: el corte cae en la banda superior (convención del curso)
    return pd.cut(score, [-np.inf] + CORTES_BANDA + [np.inf], labels=ETIQUETAS,
                  right=False)

print(f"TC {TC:.4f} · delta {delta:+.4f} · tu modelo calibrado está en producción: "
      f"hoy lo validamos")

---
# Parte A · Validación y monitoreo
### Clase 5 · 12 puntos

¿Tu modelo sigue siendo verdad? Discriminación con intervalos, estabilidad, backtesting de
calibración y el tablero de monitoreo que un validador independiente firmaría.

**Cómo funciona esta parte.** En cada ítem, la celda **«mecánica»** ya trae el cálculo de la
clase 5 aplicado a **tu** modelo y **tu** muestra (no la modifiques: corre sola). Lo que se
puntúa es la celda **«👇 TU LECTURA»**: qué dicen tus números y qué decides con ellos. Un
texto que empieza con «ESCRIBE AQUÍ» cuenta como no hecho; el ✅ aparece cuando la lectura
está escrita, y el corrector la lee con la pauta. Los umbrales del curso están declarados en
cada ítem; si propones otros, decláralos.

---
## 1. ¿Sigue ordenando? AUC, Gini y KS por muestra (2 puntos)

La mecánica calcula AUC, Gini (`2·AUC − 1`) y KS (`max(TPR − FPR)`) de tu modelo en DEV, HO y
OOT sobre la **PD calibrada**, y la caída relativa del Gini respecto de DEV. Convención del
curso para la caída: **> 20 % amarillo, > 30 % rojo**.

In [ ]:
# ---- Mecánica (NO modificar): métricas de discriminación por muestra ----
from scipy import stats as _st


def metricas_una_muestra(pd_pred, malo):
    auc = roc_auc_score(malo, pd_pred)
    fpr, tpr, _ = roc_curve(malo, pd_pred)
    return {"auc": auc, "gini": 2 * auc - 1, "ks": float(np.max(tpr - fpr))}


metricas = pd.DataFrame({m: metricas_una_muestra(PD_CAL[m], MUESTRAS[m]["malo"].values)
                         for m in ("DEV", "HO", "OOT")}).T
metricas["caida_gini_rel"] = 1 - metricas["gini"] / metricas.loc["DEV", "gini"]
print(metricas.round(4).to_string())

# controles internos: el δ no cambia el ranking y el KS coincide con scipy
for _m in ("DEV", "HO", "OOT"):
    assert abs(metricas.loc[_m, "gini"] - (2 * roc_auc_score(MUESTRAS[_m]["malo"], PD_RAW[_m]) - 1)) < 1e-9
    _ks_scipy = _st.ks_2samp(SCORE_CAL[_m][MUESTRAS[_m]["malo"].values == 1],
                             SCORE_CAL[_m][MUESTRAS[_m]["malo"].values == 0]).statistic
    assert abs(metricas.loc[_m, "ks"] - _ks_scipy) < 1e-6
caida_oot = float(metricas.loc["OOT", "caida_gini_rel"])
semaforo_caida = "🔴" if caida_oot > 0.30 else ("🟡" if caida_oot > 0.20 else "🟢")
print(f"\ncaída relativa del Gini DEV→OOT: {caida_oot:.1%} → {semaforo_caida} (umbrales 20 % / 30 %)")

In [ ]:
LECTURA_1B = """ESCRIBE AQUÍ: separa las dos caídas (DEV→HO y HO→OOT): cuál es optimismo de selección y cuál deterioro temporal; qué semáforo enciende TU caída con la convención del curso y por qué"""

# ---- Verificación (NO modificar) ----
if completo(LECTURA_1B):
    print("✅ 1 — métricas calculadas y lectura escrita (el contenido lo revisa la pauta)")
else:
    print("⏭️ Ítem 1 a medias: los números están; falta tu LECTURA_1B.")

---
## 2. La tabla de deciles/ganancia (1,5 puntos)

La mecánica ordena OOT por score calibrado en 10 deciles (decil 1 = peor score) y muestra por
decil la tasa de malos, la captura acumulada de malos, la población acumulada y el lift.

In [ ]:
# ---- Mecánica (NO modificar): deciles y ganancia sobre OOT ----
df_dec = pd.DataFrame({"malo": oot["malo"].values, "score": SCORE_CAL["OOT"]})
df_dec["decil"] = pd.qcut(df_dec["score"], 10, labels=False, duplicates="drop") + 1
deciles = df_dec.groupby("decil").agg(
    n=("malo", "size"), malos=("malo", "sum"), tasa_malos=("malo", "mean"))
deciles["malos"] = deciles["malos"].astype(int)
deciles["captura_malos"] = deciles["malos"].cumsum() / df_dec["malo"].sum()
deciles["pob_acum"] = deciles["n"].cumsum() / len(df_dec)
deciles["lift_acum"] = deciles["captura_malos"] / deciles["pob_acum"]
print(deciles.round(4).to_string())
print(f"\ntasa global OOT: {df_dec['malo'].mean():.2%} · el peor decil captura "
      f"{deciles['captura_malos'].iloc[0]:.1%} de los malos (lift {deciles['lift_acum'].iloc[0]:.1f})")
assert abs(deciles["captura_malos"].iloc[-1] - 1.0) < 1e-9
if len(deciles) < 10:
    print(f"ℹ️ {len(deciles)} grupos por empates de score: es normal, coméntalo")

In [ ]:
LECTURA_2B = """ESCRIBE AQUÍ: qué porcentaje de los malos concentra tu peor decil (y los dos peores), qué significa el lift para una política que rechaza abajo de un corte, y si la tasa de malos baja de forma ordenada de decil en decil"""

# ---- Verificación (NO modificar) ----
if completo(LECTURA_2B):
    print("✅ 2 — deciles calculados y lectura escrita")
else:
    print("⏭️ Ítem 2 a medias: la tabla está; falta tu LECTURA_2B.")

---
## 3. Bootstrap: el intervalo de tu Gini (2 puntos)

La mecánica es la de la demo: 1.000 réplicas con reemplazo del Gini en HO y OOT (semillas fijas
y distintas por muestra, para que las réplicas no queden acopladas) e IC 95 % por percentiles de
cada Gini **y de la caída HO−OOT**. Tarda unos 30–40 s en Colab.

In [ ]:
# ---- Mecánica (NO modificar): bootstrap del Gini en HO y OOT ----
def bootstrap_gini(pd_pred, malo, B=1000, semilla=20260916):
    y = np.asarray(malo, dtype=float)
    p = np.asarray(pd_pred, dtype=float)
    rng = np.random.default_rng(semilla)
    ginis = np.empty(B)
    for b in range(B):
        while True:                       # réplica sin ambas clases: se repite
            idx = rng.integers(0, len(y), len(y))
            yb, pb = y[idx], p[idx]
            if 0 < yb.sum() < len(yb):
                break
        ginis[b] = 2 * roc_auc_score(yb, pb) - 1
    return ginis


BOOT = {m: bootstrap_gini(PD_CAL[m], MUESTRAS[m]["malo"].values, semilla=sem)
        for m, sem in (("HO", 20260916), ("OOT", 20260917))}
IC = {}
for m in BOOT:
    IC[m] = tuple(np.percentile(BOOT[m], [2.5, 97.5]))
    print(f"{m}: Gini {metricas.loc[m, 'gini']:.3f} · IC95 [{IC[m][0]:.3f}, {IC[m][1]:.3f}]")
    assert IC[m][0] < metricas.loc[m, "gini"] < IC[m][1]
caida_ho_oot = float(metricas.loc["HO", "gini"] - metricas.loc["OOT", "gini"])
IC["caida"] = tuple(np.percentile(BOOT["HO"] - BOOT["OOT"], [2.5, 97.5]))
contiene_cero = IC["caida"][0] <= 0 <= IC["caida"][1]
print(f"caída HO−OOT: {caida_ho_oot:+.3f} · IC95 [{IC['caida'][0]:+.3f}, {IC['caida'][1]:+.3f}] → "
      f"{'contiene el 0: NO concluyente' if contiene_cero else 'no contiene el 0: caída demostrable'}")

In [ ]:
LECTURA_3B = """ESCRIBE AQUÍ: con TU intervalo: la caída HO−OOT ¿es concluyente o cabe en el ruido de la muestra?; qué le responderías a un gerente que pide «congelar el modelo hasta que se aclare» y qué decides tú (con el costo de cada opción)"""

# ---- Verificación (NO modificar) ----
if completo(LECTURA_3B):
    print("✅ 3 — bootstrap calculado y lectura escrita")
else:
    print("⏭️ Ítem 3 a medias: los intervalos están; falta tu LECTURA_3B.")

---
## 4. Estabilidad: PSI del score y CSI de tus variables (2 puntos)

La mecánica calcula el **PSI del score** sobre las 8 bandas de la master scale (DEV→OOT y
DEV→TTD, con el aporte de cada banda) y el **CSI** de tus `ELEGIDAS` (DEV→TTD), de mayor a
menor. Umbrales del curso: **0,10 amarillo, 0,25 rojo**.

In [ ]:
# ---- Mecánica (NO modificar): PSI por bandas y CSI por variable ----
def psi_por_bandas(score_esperado, score_actual, suavizado=1e-4):
    """PSI sobre las 8 bandas de la master scale, con el aporte por banda."""
    e = pd.Series(banda_de(score_esperado)).value_counts(normalize=True).reindex(ETIQUETAS).fillna(0)
    a = pd.Series(banda_de(score_actual)).value_counts(normalize=True).reindex(ETIQUETAS).fillna(0)
    pe, pa = e + suavizado, a + suavizado
    tabla = pd.DataFrame({"pct_esperado": e, "pct_actual": a,
                          "aporte": (pa - pe) * np.log(pa / pe)})
    return float(tabla["aporte"].sum()), tabla


def csi_de(variable, base_df, actual_df, suavizado=1e-4):
    """CSI: PSI sobre los bins WoE del scorecard (cortes de DEV)."""
    e_lab, _ = binear(base_df[variable])
    a_lab, _ = binear(actual_df[variable], ref=base_df[variable])
    e = e_lab.value_counts(normalize=True)
    a = a_lab.value_counts(normalize=True)
    cats = sorted(set(e.index) | set(a.index))
    pe = e.reindex(cats).fillna(0) + suavizado
    pa = a.reindex(cats).fillna(0) + suavizado
    return float(((pa - pe) * np.log(pa / pe)).sum())


psi_oot, tabla_psi_oot = psi_por_bandas(SCORE_CAL["DEV"], SCORE_CAL["OOT"])
psi_ttd, tabla_psi_ttd = psi_por_bandas(SCORE_CAL["DEV"], SCORE_CAL["TTD"])
sem_psi = lambda v: "🔴" if v > 0.25 else ("🟡" if v > 0.10 else "🟢")
print(f"PSI del score DEV→OOT: {psi_oot:.4f} {sem_psi(psi_oot)} · DEV→TTD: {psi_ttd:.4f} {sem_psi(psi_ttd)}\n")
print("aportes por banda DEV→TTD (de la mejor a la peor banda):")
print(tabla_psi_ttd.iloc[::-1].round(4).to_string())

csi = pd.Series({v: csi_de(v, dev, ttd) for v in ELEGIDAS}).sort_values(ascending=False)
print("\nCSI DEV→TTD por variable:")
print(csi.round(4).to_string())
print(f"\nCSI máximo: {csi.max():.3f} ({csi.idxmax()}) {sem_psi(csi.max())}")
assert abs(tabla_psi_ttd["aporte"].sum() - psi_ttd) < 1e-9

In [ ]:
LECTURA_4B = """ESCRIBE AQUÍ: qué bandas explican tu PSI DEV→TTD y hacia dónde se movió el flujo (¿hacia bandas mejores o peores?); si alguna variable cruzó el 0,10 de CSI; y si el corrimiento es coherente con el deterioro macro de 2025"""

# ---- Verificación (NO modificar) ----
if completo(LECTURA_4B):
    print("✅ 4 — PSI y CSI calculados y lectura escrita")
else:
    print("⏭️ Ítem 4 a medias: PSI y CSI están; falta tu LECTURA_4B.")

---
## 5. Backtesting de calibración: el momento de la verdad (2,5 puntos)

En la parte B del Lab 2 tu calibración dejó un residuo en OOT y prometiste vigilarlo con un test.
Ese momento es ahora. La mecánica corre sobre **OOT** el test binomial **global** (malos
observados contra la PD calibrada media) y el binomial **por banda** de la master scale, con los
alfas del curso (**0,05 amarillo, 0,01 rojo**).

In [ ]:
# ---- Mecánica (NO modificar): binomial global y por banda sobre OOT ----
from scipy import stats

n_oot, malos_oot = len(oot), int(oot["malo"].sum())
pd_media = float(PD_CAL["OOT"].mean())
p_global = stats.binomtest(malos_oot, n_oot, pd_media).pvalue
residuo_pts = (oot["malo"].mean() - pd_media) * 100
print(f"GLOBAL: {malos_oot} malos observados vs {n_oot * pd_media:.1f} esperados "
      f"(tasa {oot['malo'].mean():.2%} vs PD media {pd_media:.2%}, residuo {residuo_pts:+.1f} pts) "
      f"→ p = {p_global:.5f} {'🔴' if p_global < 0.01 else ('🟡' if p_global < 0.05 else '🟢')}")

bt = pd.DataFrame({"banda": banda_de(SCORE_CAL["OOT"]), "pd": PD_CAL["OOT"],
                   "malo": oot["malo"].values}).groupby("banda", observed=False).agg(
    n=("malo", "size"), malos=("malo", "sum"), pd_calibrada=("pd", "mean"),
    tasa_observada=("malo", "mean"))
bt["malos"] = bt["malos"].astype(int)
# una banda vacía (n=0) no se testea: queda NaN, jamás cuenta como verde
bt["p_valor"] = [stats.binomtest(int(f["malos"]), int(f["n"]), f["pd_calibrada"]).pvalue
                 if f["n"] > 0 else np.nan
                 for _, f in bt.iterrows()]
bt["semaforo"] = np.where(bt["p_valor"].isna(), "—",
                 np.where(bt["p_valor"] < 0.01, "🔴",
                 np.where(bt["p_valor"] < 0.05, "🟡", "🟢")))
print("\n" + bt.iloc[::-1].round(4).to_string())
bandas_rojas = [str(b) for b in bt.index[bt["p_valor"] < 0.01]]
bandas_amarillas = [str(b) for b in bt.index[(bt["p_valor"] >= 0.01) & (bt["p_valor"] < 0.05)]]
subestimadas = int((bt.loc[bt["n"] > 0, "tasa_observada"] > bt.loc[bt["n"] > 0, "pd_calibrada"]).sum())
print(f"\nbandas 🔴: {bandas_rojas or 'ninguna'} · 🟡: {bandas_amarillas or 'ninguna'} · "
      f"bandas donde se observa más mora que la prometida: {subestimadas} de {int((bt['n'] > 0).sum())}")

In [ ]:
LECTURA_5B = """ESCRIBE AQUÍ: (i) por qué TU residuo sí es significativo (n, magnitud del residuo, deterioro de Andes) a diferencia de la demo de Austral (p 0,56); (ii) qué acción proporcionada corresponde según la jerarquía de la clase (vigilar → recalibrar δ → re-desarrollar → contingencia); (iii) por qué esa y no una más drástica: qué dicen tus ítems 1–3 sobre el ranking"""

# ---- Verificación (NO modificar) ----
if completo(LECTURA_5B, minimo=150):
    print("✅ 5 — backtesting calculado y contraste escrito")
else:
    print("⏭️ Ítem 5 a medias: los tests están; falta tu LECTURA_5B con sus tres partes.")

---
## 6. TU tablero de monitoreo (2 puntos)

El entregable que va directo a tu informe de la E3. La mecánica arma el tablero con **ocho
indicadores** tomados de los ítems anteriores, su valor de hoy, el umbral del curso, el semáforo
calculado y la frecuencia; incluye un indicador **definido hoy para el futuro** (la TC realizada
contra el ancla, que aún no puede medirse).

**Lo que pones tú:** para **cada indicador en 🔴** (y los 🟡 si quieres), la **acción** y el
**dueño** en `ACCIONES` (usa el `id` que imprime la tabla), y el diagnóstico 6b.

In [ ]:
# ---- Mecánica (NO modificar): el tablero con los umbrales del curso ----
mix_de_mod = float(pd.Series(banda_de(np.concatenate(
    [SCORE_CAL["DEV"], SCORE_CAL["HO"], SCORE_CAL["OOT"]]))).value_counts(
    normalize=True)[["D", "E"]].sum())
mix_de_ttd = float(pd.Series(banda_de(SCORE_CAL["TTD"])).value_counts(
    normalize=True)[["D", "E"]].sum())
n_fuera = int((bt["p_valor"] < 0.05).sum())
n_rojas = int((bt["p_valor"] < 0.01).sum())

sem_mayor = lambda v, a, r: "🔴" if v > r else ("🟡" if v > a else "🟢")
sem_menor = lambda v, a, r: "🔴" if v < r else ("🟡" if v < a else "🟢")

tablero = pd.DataFrame([
    ("gini", "Gini OOT (caída vs DEV)", f"{metricas.loc['OOT', 'gini']:.3f} (−{caida_oot:.1%})",
     "caída > 20% / > 30%", sem_mayor(caida_oot, 0.20, 0.30), "mensual"),
    ("ks", "KS OOT", f"{metricas.loc['OOT', 'ks']:.3f}", "< 0,30 / < 0,20",
     sem_menor(metricas.loc["OOT", "ks"], 0.30, 0.20), "mensual"),
    ("psi", "PSI score DEV→TTD (8 bandas)", f"{psi_ttd:.3f}", "> 0,10 / > 0,25",
     sem_mayor(psi_ttd, 0.10, 0.25), "mensual"),
    ("csi", f"CSI máximo ({csi.idxmax()})", f"{csi.max():.3f}", "> 0,10 / > 0,25",
     sem_mayor(csi.max(), 0.10, 0.25), "mensual"),
    ("binomial", "Binomial global OOT", f"p = {p_global:.4f}", "p < 0,05 / < 0,01",
     sem_menor(p_global, 0.05, 0.01), "trimestral"),
    ("bandas", "Bandas binomial fuera (OOT)", f"{n_fuera - n_rojas} 🟡 · {n_rojas} 🔴",
     "≥ 1 🟡 / ≥ 1 🔴", "🔴" if n_rojas else ("🟡" if n_fuera else "🟢"), "trimestral"),
    ("mix", "Mix D+E en bandeja TTD",
     f"{mix_de_ttd:.1%} ({(mix_de_ttd - mix_de_mod) * 100:+.1f} pts)",
     "> +3 / > +6 pts", sem_mayor((mix_de_ttd - mix_de_mod) * 100, 3, 6), "mensual"),
    ("tc_futura", "TC realizada vs ancla (post-recalibración)",
     "por medir: primer corte con cosechas 2025-H2 maduras (jul 2027)",
     "desvío > 1 / > 2 pts", "⏳", "trimestral"),
], columns=["id", "indicador", "valor_hoy", "umbral", "estado", "frecuencia"]).set_index("id")
print(tablero.to_string())
_conteo = tablero["estado"].value_counts()
print(f"\nresumen: {_conteo.get('🟢', 0)} 🟢 · {_conteo.get('🟡', 0)} 🟡 · "
      f"{_conteo.get('🔴', 0)} 🔴 · {_conteo.get('⏳', 0)} ⏳")
ids_rojos = list(tablero.index[tablero["estado"] == "🔴"])
print("indicadores en 🔴 que necesitan acción y dueño:", ids_rojos or "ninguno")

In [ ]:
ACCIONES = {
    # un registro por cada id en 🔴 (agrega los 🟡 si quieres): {"accion": "...", "dueño": "..."}
    "binomial": {"accion": ..., "dueño": ...},
}
LECTURA_6B = """ESCRIBE AQUÍ: el diagnóstico para el comité: (i) el patrón que muestran tus semáforos, (ii) la acción que recomiendas con su gatillo cumplido, (iii) qué NO recomiendas hacer y por qué, (iv) quién debería firmar. Si usaste la lista ELEGIDAS de referencia, decláralo"""

# ---- Verificación (NO modificar) ----
faltan_rojos = [i for i in ids_rojos
                if not (isinstance(ACCIONES.get(i), dict)
                        and completo(ACCIONES[i].get("accion", ...), minimo=15)
                        and completo(ACCIONES[i].get("dueño", ...), minimo=3))]
if faltan_rojos or not completo(LECTURA_6B, minimo=150):
    print("⏭️ Ítem 6 a medias:", ("faltan acción y dueño para " + ", ".join(faltan_rojos) + " · " if faltan_rojos else "")
          + ("falta tu LECTURA_6B (diagnóstico)." if not completo(LECTURA_6B, minimo=150) else ""))
else:
    tablero["accion"] = [ACCIONES.get(i, {}).get("accion", "") for i in tablero.index]
    tablero["dueño"] = [ACCIONES.get(i, {}).get("dueño", "") for i in tablero.index]
    print(tablero[["indicador", "valor_hoy", "estado", "accion", "dueño"]].to_string())
    print(f"\n✅ 6 — tablero con acción y dueño en los {len(ids_rojos)} indicador(es) en 🔴 y diagnóstico escrito")

---
# Parte B · Implementación y gobierno
### Clase 6 · 15 puntos · **elige UNA de las dos opciones**

El modelo de la parte A ya está validado. La parte B lo lleva a producción con evidencia: corrida
reproducible, trazabilidad, model card, gatillos y resumen ejecutivo. Puedes hacerlo de dos formas,
**con el mismo puntaje máximo y la misma pauta**:

| Opción | Qué haces | Para quién |
|---|---|---|
| **1 · con `nikodym`** (B1–B5) | La corrida completa que viste en la clase 6, sobre tu muestra. La mecánica viene escrita; tú pones la regla del contrato, el cutoff, y escribes lecturas, limitaciones, gatillos y resumen | Quien quiere concentrarse en el criterio y no en el código |
| **2 · a mano** (tareas 7–12) | Construyes tú las piezas: artefacto congelado, contrato de datos, corrida con audit trail, model card, gatillos y resumen | Quien quiere ver cada pieza por dentro |

Haz **una sola**. La opción que no hagas queda en ⏭️ y no se corrige; si entregas las dos, se
corrige la de `nikodym`.

---
# Opción 1 · Parte B con `nikodym` (B1–B5)

Si elegiste la opción 2 (a mano), salta hasta **Opción 2**: las celdas de esta opción igual corren
solas con «Reiniciar y ejecutar todo» (instalan `nikodym` y descargan la matriz) y sus tareas quedan
en ⏭️; no te afectan.

Esta opción repite **con la librería** la corrida que viste en la clase 6 con Banco Austral, ahora
sobre **tu** muestra de Financiera Andes: de la matriz modelable al scorecard, las métricas, el
audit trail, el lineage, la model card y el informe. Lo que nikodym automatiza es cálculo y
evidencia; la decisión, los límites y los gatillos siguen siendo tuyos — y eso es lo que se puntúa.

| Tarea | Qué pones tú (lo que se puntúa) | Qué viene escrito | Puntos |
|---|---|---|---:|
| **B1** | Una regla de rango propia y su justificación (la TC la toma sola de tu calibración de la celda 0) | La config F1 completa y la validación del contrato | 3 |
| **B2** | La lectura de las tres primeras decisiones del trail | La corrida, los identificadores y la lectura del archivo | 3 |
| **B3** | Un juicio sobre las diferencias con tu modelo de la parte A | Scorecard, Gini, deciles, PSI y calibración, comparados con tu ítem 1 y tus `ELEGIDAS` | 3 |
| **B4** | Propósito, supuestos, ≥ 3 limitaciones propias y tu juicio sobre las automáticas | La ficha generada y guardada | 3 |
| **B5** | Tu cutoff con su justificación, tres gatillos y el resumen ejecutivo | Secciones del informe, tabla de impacto del cutoff, estado de los gatillos, archivos | 3 |

**Cómo se lee cada tarea:** la celda `👇 TU CÓDIGO` tiene arriba un bloque de **insumos** (variables
con `...` o textos que empiezan con «ESCRIBE AQUÍ») y abajo la mecánica ya escrita. Mientras un
insumo falte, la celda imprime ⏭️ y no falla. El ✅ de cada tarea confirma que está **completa**, no
que esté bien: el contenido lo lee el corrector con la pauta.

## Fijar el entorno

El pin fija la versión validada en clase (`nikodym==1.11.0`). Sin el `==`, Colab instalaría
la última versión y el mismo notebook podría producir otro pipeline u otro informe. La celda
muestra la versión efectiva antes de tocar los datos. `esquema_de` es la misma función de la clase:
lee el contrato de entrada (nombre, tipo, nulabilidad) de la matriz misma.

In [ ]:
!pip install -q "nikodym[scoring]==1.11.0"

import copy
import json
import operator
import re
import shutil
import urllib.request
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq
import nikodym

from IPython.display import Markdown
from nikodym import check_dataset
from nikodym.audit import read_trail
from nikodym.calibration import CalibrationConfig
from nikodym.core.config import NikodymConfig, config_hash
from nikodym.data import DataConfig
from nikodym.governance import GovernanceConfig, ModelCardBuilder
from nikodym.report import ReportConfig
from nikodym.ui.presets import standard_preset

PIN_NIKODYM = "1.11.0"
URL_MATRIZ = "https://raw.githubusercontent.com/nexolabs-gh/datos-riesgo-credito/main/andes_matriz_modelable.parquet"
SALIDA = Path("salida_lab3b")
TRAIL = SALIDA / "audit_trail_andes.jsonl"
MAPA_MUESTRAS = {"desarrollo": "DEV", "holdout": "HO", "oot": "OOT"}
MIN_MALOS_CATEGORIA = 8
METADATOS = {"id_cliente", "fecha_solicitud", "cohorte", "muestra", "malo", "indeterminado"}


def esquema_de(matriz):
    """El contrato de entrada, leído de la matriz (como en la clase): nombre, tipo y si admite nulos."""
    def tipo(serie):
        if serie.name == "malo" or pd.api.types.is_float_dtype(serie):
            return "float"
        if pd.api.types.is_integer_dtype(serie):
            return "int"
        return "bool" if pd.api.types.is_bool_dtype(serie) else "str"
    esquema = []
    for columna in matriz.columns:
        regla = {"name": columna, "dtype": tipo(matriz[columna])}
        if columna == "malo" or columna not in METADATOS:
            regla["nullable"] = True          # el target y las candidatas pueden venir vacíos
        esquema.append(regla)
    predictoras = [c for c in matriz.columns if c not in METADATOS]
    categoricas = [c for c in predictoras if pd.api.types.is_object_dtype(matriz[c])]
    return esquema, predictoras, categoricas


assert nikodym.__version__ == PIN_NIKODYM, f"se instaló {nikodym.__version__}; el lab exige {PIN_NIKODYM}"
print(f"nikodym {nikodym.__version__} · entorno listo")

## Tu matriz modelable

Tu muestra personal (`mis_clientes`) ya existe desde la celda 0. Aquí se descarga la matriz modelable
publicada (las 108 candidatas de la fábrica del curso, ya ancladas al punto de observación), se
filtra a tu muestra y se guarda `mi_matriz_andes.parquet`, el archivo que `nikodym` va a leer. Es la
única celda de la parte B que prepara datos con pandas.

Al final deja **a la vista** una decisión que en la clase se tomó a mano con `estado_civil`: una
categoría con muy pocos malos puede quedar **sin malos** en la muestra de desarrollo que sortea la
librería, y el binning estricto de `nikodym` se detiene («WoE no defendible por bin con una clase en
cero»). Por eso las categóricas con alguna categoría de menos de `MIN_MALOS_CATEGORIA` malos en tu
población de ajuste se dejan **fuera de la lista de candidatas** — la matriz no se toca y la
exclusión queda impresa para que la cites en B1.

In [ ]:
# ---- Matriz modelable publicada, filtrada a tu muestra (NO modificar) ----
RUTA_MATRIZ = Path("andes_matriz_modelable.parquet")
RUTA_PERSONAL = Path("mi_matriz_andes.parquet")
urllib.request.urlretrieve(URL_MATRIZ, RUTA_MATRIZ)
matriz = pd.read_parquet(RUTA_MATRIZ)
ESQUEMA_BASE, PREDICTORAS_MATRIZ, CATEGORICAS_MATRIZ = esquema_de(matriz)   # el contrato, como en C6
mi_matriz = matriz[matriz["id_cliente"].isin(mis_clientes)].copy()
mi_matriz.to_parquet(RUTA_PERSONAL, index=True)
assert mi_matriz.index.name == "id_solicitud" and mi_matriz.index.is_unique

# Población de ajuste: la misma definición de la celda 0 (con desempeño y no indeterminada).
ajustable = mi_matriz[
    mi_matriz["muestra"].isin(["DEV", "HO", "OOT"])
    & mi_matriz["malo"].notna()
    & ~mi_matriz["indeterminado"]
]
TC_CONTROL = float(ajustable.groupby("cohorte", sort=True)["malo"].mean().mean())

# Candidatas que el binning estricto no puede defender en TU muestra (exclusión visible).
no_oot = ajustable[ajustable["muestra"].isin(["DEV", "HO"])]
CANDIDATAS_EXCLUIDAS = {}
for columna in CATEGORICAS_MATRIZ:
    malos_por_categoria = no_oot.groupby(columna)["malo"].sum()
    raras = malos_por_categoria[malos_por_categoria < MIN_MALOS_CATEGORIA]
    if len(raras):
        CANDIDATAS_EXCLUIDAS[columna] = {k: int(v) for k, v in raras.items()}
PREDICTORAS = [c for c in PREDICTORAS_MATRIZ if c not in CANDIDATAS_EXCLUIDAS]
CATEGORICAS = [c for c in CATEGORICAS_MATRIZ if c in PREDICTORAS]

print(f"Tu matriz: {len(mi_matriz):,} filas · índice {mi_matriz.index.name} · "
      f"{len(ajustable):,} con desempeño para ajustar")
print(mi_matriz["muestra"].value_counts().reindex(["DEV", "HO", "OOT", "TTD"]).to_string())
print(f"TC de control (promedio simple de cohortes): {TC_CONTROL:.4%} · tu TC del Lab 2 · parte B: {TC:.4%}")
print(f"\nCandidatas que entran al binning: {len(PREDICTORAS)} de {len(PREDICTORAS_MATRIZ)}")
for columna, raras in CANDIDATAS_EXCLUIDAS.items():
    print(f" · fuera: {columna} — categorías con menos de {MIN_MALOS_CATEGORIA} malos en tu "
          f"población de ajuste: {raras}")

## Herramientas de la parte B (NO modificar)

Cinco funciones pequeñas para que las tareas se concentren en la decisión y no en la API:
`objetos_de_comite` recupera las tablas de la corrida, `impacto_cutoff` mide aprobación y mora
por cutoff, `evaluar_gatillos` calcula el estado de cada gatillo, `tabla_markdown` escribe una
tabla y `completo` distingue un insumo escrito de un placeholder.

In [ ]:
PLACEHOLDER = "ESCRIBE AQUÍ"


def completo(*insumos, minimo=60):
    """True si ningún insumo es `...`, ni un placeholder, ni un texto demasiado corto."""
    for insumo in insumos:
        if insumo is ...:
            return False
        if isinstance(insumo, str):
            if insumo.strip().startswith(PLACEHOLDER) or len(insumo.strip()) < minimo:
                return False
        elif isinstance(insumo, (list, tuple, dict)):
            if len(insumo) == 0 or any(v is ... for v in (insumo.values() if isinstance(insumo, dict) else insumo)):
                return False
    return True


def objetos_de_comite(study):
    """Las tablas que un comité pide, tal como las publicó cada etapa de la corrida."""
    metricas_ = study.artifacts.get("performance", "discriminant_metrics").copy()
    metricas_["muestra"] = metricas_["partition"].map(MAPA_MUESTRAS)
    deciles = study.artifacts.get("performance", "performance_table").copy()
    estabilidad = study.artifacts.get("stability", "stability_metrics").copy()
    calibracion = study.artifacts.get("validation", "calibration").copy()
    return {
        "scorecard": study.artifacts.get("scorecard", "scorecard"),
        "metricas": metricas_.set_index("muestra")[["n_total", "n_bad", "auc", "gini", "ks"]],
        "deciles_oot": deciles.loc[deciles["partition"].eq("oot"),
                                   ["decile", "n_total", "n_bad", "bad_rate", "mean_pd",
                                    "min_score", "max_score", "lift"]],
        "psi": estabilidad.loc[estabilidad["metric"].eq("score_psi"),
                               ["comparison", "value", "band", "action"]],
        "csi": estabilidad.loc[estabilidad["metric"].eq("csi")
                               & estabilidad["comparison"].eq("dev_vs_oot"),
                               ["feature", "value", "band"]].sort_values("value", ascending=False),
        "calibracion": calibracion[["partition", "test", "n", "observed_defaults", "expected_pd",
                                    "observed_dr", "statistic", "p_value", "decision"]],
        "overall_status": study.artifacts.get("validation", "result").card.overall_status,
    }


def impacto_cutoff(study, cutoffs, particiones=("holdout", "oot")):
    """Aprobación, mora observada y PD calibrada media de los aprobados, por cutoff y muestra."""
    score = study.artifacts.get("scorecard", "score")[["partition", "target", "score"]]
    pd_cal = study.artifacts.get("calibration", "calibrated_pd_frame")[["pd_calibrated"]]
    base = score.join(pd_cal)
    filas = []
    for particion in particiones:
        datos = base[base["partition"].eq(particion)]
        for cutoff in cutoffs:
            aprobados = datos[datos["score"] >= cutoff]
            filas.append({
                "muestra": MAPA_MUESTRAS[particion], "cutoff": int(cutoff), "n": len(datos),
                "aprobacion": len(aprobados) / len(datos),
                "malos_obs_aprobados": float(aprobados["target"].mean()),
                "pd_cal_media_aprobados": float(aprobados["pd_calibrated"].mean()),
            })
    return pd.DataFrame(filas)


def tabla_markdown(df):
    """Tabla Markdown sin dependencias (Colab no siempre trae tabulate)."""
    columnas = list(df.columns)
    lineas = ["| " + " | ".join(columnas) + " |", "|" + "|".join(["---"] * len(columnas)) + "|"]
    for _, fila in df.iterrows():
        lineas.append("| " + " | ".join(str(fila[c]) for c in columnas) + " |")
    return "\n".join(lineas)


OPERADORES = {"<": operator.lt, "<=": operator.le, ">": operator.gt, ">=": operator.ge}


def evaluar_gatillos(gatillos, valores_hoy):
    """`disparado` se calcula contra el valor de hoy; nunca se escribe a mano."""
    filas = []
    for gatillo in gatillos:
        metrica = gatillo.get("metrica")
        if metrica is None:
            condicion, valor_hoy, disparado = gatillo["condicion"], "sin insumo en esta corrida", "no evaluable"
        else:
            valor = float(valores_hoy[metrica])
            condicion = f"{metrica} {gatillo['operador']} {gatillo['umbral']}"
            valor_hoy = f"{valor:.4f}"
            disparado = "SÍ" if OPERADORES[gatillo["operador"]](valor, gatillo["umbral"]) else "no"
        filas.append({"gatillo": gatillo["gatillo"], "condicion": condicion, "valor_hoy": valor_hoy,
                      "disparado": disparado, "responsable": gatillo["responsable"],
                      "accion": gatillo["accion"]})
    return pd.DataFrame(filas).set_index("gatillo")


print("herramientas listas: completo · objetos_de_comite · impacto_cutoff · evaluar_gatillos · tabla_markdown")

---
## B1 · Config y contrato — 3 puntos

La config parte del **preset F1** de la clase y declara lo mismo que declaramos para Austral:
matriz e índice `id_solicitud`, target `malo` (1 = malo; `NaN` = sin desempeño, excluido del
ajuste), indeterminados fuera del ajuste, cohortes OOT marzo–junio de 2025, holdout 0,3, informe
en HTML y Markdown, audit trail y gobierno. Todo eso viene escrito. La **TC** que ancla la
calibración es la de tu Lab 2 · parte B, calculada en la celda 0 (`TC`); si allá usaste otra,
cámbiala.

**Lo que pones tú (y se puntúa):**

1. `REGLA_PROPIA`: una regla de rango (`ge` y/o `le`) sobre una columna numérica de la matriz,
   que exprese una condición de **tu política** (la de referencia es la antigüedad mínima).
2. `JUSTIFICACION_B1`: por qué esa regla, **qué bloquea y qué solo avisa**, y qué candidatas
   quedaron fuera de la lista (las imprimió la celda anterior).

Dos puertas distintas, no confundirlas: `check_dataset` **avisa** antes de correr si faltan
columnas o el índice no calza; las reglas `ge`/`le` viven dentro de `run` y **bloquean** la corrida
completa si una fila las viola (`DataValidationError`, como el `config_roto` de la clase). La celda
te dice cuántas filas de tu matriz violarían tu regla: con cero, B2 corre; con más de cero, B2 se
detiene — ajusta la regla a tu población o explica en B1 por qué prefieres el bloqueo.

**Salida esperada:** `compatible: True`, cero `mismatches`, `config_hash`, el rango observado de
la columna elegida y cero filas fuera de rango.

In [ ]:
# 👇 TU CÓDIGO — insumos
TC_LAB2B = TC                       # tu TC del Lab 2 · parte B (celda 0); cámbiala solo si allá usaste otra
REGLA_PROPIA = {"columna": "antiguedad_meses", "ge": ..., "le": None}   # cambia la columna si tu política es otra
JUSTIFICACION_B1 = """ESCRIBE AQUÍ: por qué esta regla, qué bloquea y qué solo avisa, y qué candidatas quedaron fuera."""

# ---- mecánica (viene escrita) ----
if not completo(TC_LAB2B, JUSTIFICACION_B1) or any(v is ... for v in REGLA_PROPIA.values()):
    print("⏭️ B1 pendiente: completa REGLA_PROPIA (ge y/o le) y JUSTIFICACION_B1.")
else:
    assert 0 < TC_LAB2B < 1, f"TC_LAB2B = {TC_LAB2B}: escríbela como proporción (0,115), no en porcentaje"
    if abs(TC_LAB2B - TC_CONTROL) > 0.005:
        print(f"⚠️ tu TC ({TC_LAB2B:.4%}) difiere de la de control ({TC_CONTROL:.4%}) en más de "
              "medio punto: si no fue una decisión tuya en el Lab 2, revisa el número.")

    # 1) Contrato: el esquema publicado más TU regla de rango.
    columna_regla = REGLA_PROPIA["columna"]
    ESQUEMA = copy.deepcopy(ESQUEMA_BASE)
    especificacion = next(e for e in ESQUEMA if e["name"] == columna_regla)
    assert especificacion["dtype"] in ("int", "float"), f"{columna_regla} no es numérica"
    for limite in ("ge", "le"):
        if REGLA_PROPIA.get(limite) is not None:
            especificacion[limite] = REGLA_PROPIA[limite]
    observados = mi_matriz[columna_regla].dropna()
    minimo_regla = -float("inf") if REGLA_PROPIA.get("ge") is None else REGLA_PROPIA["ge"]
    maximo_regla = float("inf") if REGLA_PROPIA.get("le") is None else REGLA_PROPIA["le"]
    assert minimo_regla > -float("inf") or maximo_regla < float("inf"), "la regla necesita ge y/o le"
    fuera = int(((observados < minimo_regla) | (observados > maximo_regla)).sum())

    # 2) Config: preset F1 + las decisiones de Andes (las mismas de la clase, con TU TC).
    cfg = copy.deepcopy(standard_preset()["config"])
    cfg["data"]["load"]["source"] = str(RUTA_PERSONAL)
    cfg["data"]["schema"]["columns"] = ESQUEMA
    cfg["data"]["schema"]["index_col"] = "id_solicitud"
    cfg["data"]["target"]["bad_rule"] = {"all_of": [{"col": "malo", "op": "==", "value": 1}]}
    cfg["data"]["target"]["good_rule"] = None
    cfg["data"]["target"]["indeterminate_rule"] = {
        "all_of": [{"col": "indeterminado", "op": "==", "value": True}]
    }
    cfg["data"]["target"]["exclusion_rules"] = [{
        "name": "sin_desempeno",
        "rule": {"all_of": [{"col": "malo", "op": "isna"}]},
    }]
    cfg["data"]["partition"]["ttd_includes_excluded"] = True
    cfg["data"]["partition"]["strategy"] = {
        "type": "cohort",
        "cohort_col": "cohorte",
        "oot_cohorts": ["2025-03", "2025-04", "2025-05", "2025-06"],
        "holdout_fraction": 0.3,
    }
    cfg["binning"]["feature_columns"] = PREDICTORAS
    cfg["binning"]["categorical_columns"] = CATEGORICAS
    cfg["selection"]["min_iv"] = 0.10
    cfg["selection"]["correlation"]["threshold"] = 0.70
    cfg["calibration"]["target_pd"] = float(TC_LAB2B)
    cfg["calibration"]["anchor_source"] = "historical_default_rate"

    shutil.rmtree(SALIDA, ignore_errors=True)   # expediente limpio: solo archivos de ESTA sesión
    SALIDA.mkdir()
    cfg["report"]["formats"] = ["html", "md"]
    cfg["report"]["output_dir"] = str(SALIDA)
    cfg["report"]["document"] = {
        "model_name": "andes-scorecard-consumo",
        "entity": "Financiera Andes",
        "portfolio": "Crédito de consumo",
        "author": ID_ALUMNO,
        "version": "1.0 — Entrega 3",
    }
    cfg["audit"] = {"enabled": True, "trail_filename": str(TRAIL), "capture_environment": True}
    cfg["governance"] = {
        "model_name": "andes-scorecard-consumo",
        "cartera": "Consumo Financiera Andes",
        "motor": "scoring",
        "fase": "F1",
        "estado_validacion": "en_validacion",
        "author": ID_ALUMNO,
        "purpose": "Scorecard de admisión para crédito de consumo (se completa en B4).",
        "assumptions": ["Default = 90+ DPD en 12 meses."],
        "limitations": ["Modelado sobre aprobados: no observa las rechazadas."],
        "review_period_months": 12,
    }

    # 3) Validación por secciones y contrato integral.
    DataConfig.model_validate(cfg["data"])
    CalibrationConfig.model_validate(cfg["calibration"])
    ReportConfig.model_validate(cfg["report"])
    config = NikodymConfig.model_validate(cfg)
    config_hash_actual = config_hash(config)

    # 4) Primera puerta: ¿están las columnas y el índice que exige el contrato? (avisa)
    columnas_parquet = pq.read_schema(RUTA_PERSONAL).names
    revision = check_dataset(
        config,
        columns=[c for c in columnas_parquet if c != "id_solicitud"],
        index_columns=["id_solicitud"],
    )
    assert revision.compatible, revision.mismatches
    print("compatible:", revision.compatible, "· mismatches:", revision.mismatches)
    print("config_hash:", config_hash_actual)
    print(f"TC en target_pd: {TC_LAB2B:.4%} · candidatas al binning: {len(PREDICTORAS)}")
    print(f"regla propia: {columna_regla} ∈ [{minimo_regla}, {maximo_regla}] · "
          f"observado en tu matriz: [{observados.min()}, {observados.max()}] · "
          f"filas que la violan: {fuera}")
    print("\nJustificación:", JUSTIFICACION_B1)
    if fuera:
        print(f"\n⏸️ B1 en pausa: tu regla BLOQUEARÍA la corrida en B2 (segunda puerta, {fuera} filas "
              "fuera de rango). Ajusta el umbral a tu población — o, si el bloqueo es la decisión "
              "correcta, explícalo en la justificación y usa el rango observado para que B2 corra.")
    else:
        print("\n✅ B1 — config validada, contrato compatible y regla que tu matriz cumple (la calidad "
              "de la regla y de la justificación las juzga la pauta)")


---
## B2 · La corrida y su trail — 3 puntos

`nikodym.run(config)` ejecuta el pipeline completo con `audit` y `governance` encendidos. El
`run_id` identifica la corrida; `config_hash` identifica tus decisiones y `data_hash` tu insumo:
esos tres van después en el informe, la model card y el resumen (es lo que hace trazable la E3).
El trail conserva en orden los eventos y las decisiones que tomó la librería.

**Lo que pones tú:** `LECTURA_B2`, la interpretación de las **tres primeras decisiones** con los
números de tu corrida: qué regla se aplicó, sobre cuántas filas y qué significa para tu población
(pista: la primera es la exclusión de la TTD sin desempeño; la tercera cuenta lo que quedó fuera
del ajuste).

**Salida esperada:** estado `done`, tres identificadores, la cuenta de eventos por tipo, el
control de que las filas ajustadas son las de tu matriz, y tu lectura.

In [ ]:
# 👇 TU CÓDIGO — insumo
LECTURA_B2 = """ESCRIBE AQUÍ: qué hizo cada una de las tres primeras decisiones sobre TU población y por qué es correcto."""

# ---- mecánica (viene escrita) ----
if "config" not in globals():
    print("⏭️ B2 pendiente: termina B1.")
else:
    TRAIL.unlink(missing_ok=True)          # el trail es acumulativo: cada corrida parte limpio
    estudio = nikodym.run(config)
    if estudio.run_context.status != "done":
        error = estudio.run_context.error
        raise RuntimeError(f"la corrida se detuvo antes de terminar — {error.type}: "
                           f"{str(error.message)[:400]}")
    study = estudio                        # solo una corrida `done` alimenta B3–B5

    run_id = study.run_context.run_id
    lineage = study.lineage_bundle().model_dump(mode="json")
    data_hash_actual = lineage["data_hash"]
    (SALIDA / "lineage_andes.json").write_text(json.dumps(lineage, indent=2, ensure_ascii=False, default=str),
                                              encoding="utf-8")
    eventos = read_trail(TRAIL)
    decisiones = [evento for evento in eventos if evento.kind == "decision"]
    conteo_eventos = Counter(evento.kind for evento in eventos)
    assert eventos and decisiones

    # Control: lo que nikodym ajustó son exactamente tus filas con desempeño.
    tamanos = study.artifacts.get("data", "data_card").partition_sizes
    n_ajustadas = sum(tamanos[p] for p in ("desarrollo", "holdout", "oot"))
    assert n_ajustadas == len(ajustable), f"{n_ajustadas} ajustadas vs {len(ajustable)} con desempeño"

    print("status:     ", study.run_context.status)
    print("run_id:     ", run_id)
    print("config_hash:", lineage["config_hash"])
    print("data_hash:  ", data_hash_actual)
    print(f"trail: {len(eventos)} eventos · {len(decisiones)} decisiones · {dict(conteo_eventos)}")
    particiones_n = {p: tamanos[p] for p in ("desarrollo", "holdout", "oot")}
    print(f"particiones: {particiones_n} → {n_ajustadas:,} ajustadas = {len(ajustable):,} filas con desempeño ✓")
    print("\nprimeras tres decisiones (regla · umbral · valor → acción):")
    for evento in decisiones[:3]:
        p = evento.payload
        print(f" · {p['regla']} · {p['umbral']} · {p['valor']} → {p['accion']}")
    if completo(LECTURA_B2):
        print("\nLectura:", LECTURA_B2)
        print("\n✅ B2 — corrida `done`, identificadores, trail leído y lectura escrita")
    else:
        print("\n⏭️ B2 a medias: la corrida está, falta tu LECTURA_B2 de las tres decisiones.")


---
## B3 · Los objetos de comité — 3 puntos

Cuatro objetos de la **misma corrida**: scorecard, Gini/KS por muestra, PSI del score y
backtesting de calibración (Hosmer-Lemeshow por muestra). La celda los recupera, muestra además los
deciles OOT y el CSI por variable, y los **compara sola con tu parte A**: el Gini de tu ítem 1
(`metricas`) y tus variables del Lab 2 (`ELEGIDAS`).

**Lo que pones tú:** `JUICIO_B3` — ¿mismo orden de magnitud? ¿qué variables coinciden? y **al menos
dos diferencias explicadas sin llamarlas «error»** (el preset selecciona por IV y correlación y binea
con optbinning; tu Lab 2 usó una lista defendida por ti y cinco bins con `ref`).

**Salida esperada:** las tablas, la comparación de Gini, la intersección de variables y tu juicio.
Si tu ítem 1 de la parte A todavía no está, la comparación queda pendiente hasta que lo completes.

In [ ]:
# 👇 TU CÓDIGO — insumo
JUICIO_B3 = """ESCRIBE AQUÍ: orden de magnitud, variables que coinciden y al menos dos diferencias explicadas."""

# ---- mecánica (viene escrita) ----
if "study" not in globals():
    print("⏭️ B3 pendiente: termina B2.")
else:
    comite = objetos_de_comite(study)
    scorecard, metricas_comite = comite["scorecard"], comite["metricas"]
    variables_nikodym = list(dict.fromkeys(scorecard["feature"].tolist()))

    print(f"SCORECARD · {len(variables_nikodym)} variables · primeras 12 filas")
    display(scorecard[["feature", "bin_label", "woe", "beta", "points"]].head(12).round(4))
    print("\nGINI Y KS POR MUESTRA")
    display(metricas_comite.round(4))
    print("\nDECILES OOT (decil 1 = peor)")
    display(comite["deciles_oot"].round(4))
    print("\nPSI DEL SCORE")
    display(comite["psi"].round(4))
    print("\nCSI POR VARIABLE DEV→OOT (mayor primero)")
    display(comite["csi"].round(4))
    print("\nBACKTESTING DE CALIBRACIÓN (Hosmer-Lemeshow y Brier por muestra)")
    display(comite["calibracion"].round(4))
    print("overall_status de la validación formal:", comite["overall_status"])

    # Tu parte A, sin tipear nada: el Gini del ítem 1 y tus ELEGIDAS del Lab 2.
    parte_a_lista = isinstance(metricas, pd.DataFrame) and "gini" in metricas.columns
    if parte_a_lista and completo(JUICIO_B3):
        comparacion_gini = metricas_comite[["gini"]].rename(columns={"gini": "nikodym"})
        comparacion_gini["Lab 3A (a mano)"] = [float(metricas.loc[m, "gini"]) for m in comparacion_gini.index]
        comparacion_gini["diferencia"] = comparacion_gini["nikodym"] - comparacion_gini["Lab 3A (a mano)"]
        coinciden = sorted(set(ELEGIDAS) & set(variables_nikodym))
        print("\nCOMPARACIÓN CON TU PARTE A (mismo dataset, dos pipelines)")
        display(comparacion_gini.round(4))
        print("variables nikodym:", variables_nikodym)
        print(f"coinciden con tus ELEGIDAS ({len(coinciden)} de {len(ELEGIDAS)}):", coinciden)
        print("\nJuicio:", JUICIO_B3)
        print("\n✅ B3 — cuatro objetos, comparación con tu parte A y juicio escrito")
    elif not parte_a_lista:
        print("\n⏭️ B3 a medias: los objetos están; completa el ítem 1 de la parte A para comparar.")
    else:
        print("\n⏭️ B3 a medias: los objetos y la comparación están; falta tu JUICIO_B3.")


---
## B4 · Model card — 3 puntos

La ficha la genera `ModelCardBuilder` desde la corrida y el trail: identidad (`run_id`,
`config_hash`, `data_hash`), datos, supuestos, limitaciones, decisiones y fecha de revisión. La
librería **agrega sola** limitaciones que detecta en el entorno (por ejemplo, que no hay
repositorio git o falta el archivo de bloqueo de dependencias).

**Lo que pones tú:** `PROPOSITO_B4`, `SUPUESTOS_B4`, **al menos tres limitaciones propias** (cada una
con alcance y consecuencia práctica; «los datos son sintéticos» no cuenta), `MESES_REVISION` y
`JUICIO_B4`: qué limitaciones añadió la librería sola y si estás de acuerdo con declararlas.

La ficha generada **no trae la tabla de desempeño ni los usos no previstos**: esos los agregas al
documento model card de la E3 con los números de B3 (ver el anexo 1, la pauta).

**Salida esperada:** ficha ligada al mismo `run_id`, guardada en `salida_lab3b/`, tus limitaciones,
las automáticas y tu juicio.

In [ ]:
# 👇 TU CÓDIGO — insumos
PROPOSITO_B4 = """ESCRIBE AQUÍ: qué decide el modelo, para quién y con qué límite de uso."""
SUPUESTOS_B4 = [
    ...,      # p. ej. la definición de default y su ventana
    ...,      # p. ej. qué cohortes juegan de OOT
]
LIMITACIONES_PROPIAS = [
    ...,      # ≥ 3, cada una con alcance y consecuencia práctica
    ...,
    ...,
]
MESES_REVISION = ...       # cada cuántos meses se revisa
JUICIO_B4 = """ESCRIBE AQUÍ: qué limitaciones agregó la librería sola y si estás de acuerdo con declararlas."""

# ---- mecánica (viene escrita) ----
if "study" not in globals():
    print("⏭️ B4 pendiente: termina B2.")
elif not (completo(PROPOSITO_B4, JUICIO_B4) and completo(SUPUESTOS_B4, LIMITACIONES_PROPIAS, minimo=1)
          and MESES_REVISION is not ... and len(LIMITACIONES_PROPIAS) >= 3
          and all(isinstance(l, str) and len(l) > 40 for l in LIMITACIONES_PROPIAS)):
    print("⏭️ B4 pendiente: propósito, supuestos, ≥ 3 limitaciones propias (más de 40 caracteres "
          "cada una), meses de revisión y juicio.")
else:
    gobierno_b4 = copy.deepcopy(cfg["governance"])
    gobierno_b4["purpose"] = PROPOSITO_B4
    gobierno_b4["assumptions"] = SUPUESTOS_B4
    gobierno_b4["limitations"] = LIMITACIONES_PROPIAS
    gobierno_b4["review_period_months"] = int(MESES_REVISION)

    gcfg = GovernanceConfig.model_validate(gobierno_b4)
    model_card = ModelCardBuilder(gcfg).build(study, trail_path=TRAIL)
    (SALIDA / "model_card_andes.md").write_text(model_card.to_markdown(), encoding="utf-8")
    (SALIDA / "model_card_andes.json").write_text(model_card.to_json(), encoding="utf-8")
    limitaciones_automaticas = [l for l in model_card.limitations if l not in LIMITACIONES_PROPIAS]
    assert model_card.run_id == run_id and model_card.config_hash == config_hash_actual

    print("run_id:      ", model_card.run_id)
    print("config_hash: ", model_card.config_hash)
    print("data_hash:   ", model_card.data_hash)
    print("revisión:    ", model_card.review_date.date(), "→", model_card.next_review_date.date(),
          f"({MESES_REVISION} meses)")
    print("\nlimitaciones propias:")
    for limitacion in LIMITACIONES_PROPIAS:
        print(" ·", limitacion)
    print("\nlimitaciones añadidas por la librería:")
    for limitacion in limitaciones_automaticas:
        print(" ·", limitacion)
    print("\nJuicio:", JUICIO_B4)
    print(f"\n✅ B4 — model card guardada ({len(LIMITACIONES_PROPIAS)} limitaciones propias + "
          f"{len(limitaciones_automaticas)} automáticas) y juicio escrito")


---
## B5 · Informe, decisión de cutoff, gatillos y resumen ejecutivo — 3 puntos

El informe (`scorecard_report.html` y su fuente editable `scorecard_report.qmd`, Markdown válido)
ya está escrito por la corrida: 8 secciones + 3 anexos, con bloques «POR COMPLETAR» donde un
humano debe firmar. Es la **base** del informe técnico de la E3, no la entrega.

Esta tarea va en dos celdas.

**B5a — tu decisión de cutoff.** La corrida no trae estrategia: la decisión es tuya. Parte de
`CUTOFF_LAB2B` (el que recomendaste en el Lab 2 · parte B; si no lo hiciste, 540, el de referencia
del curso) y la celda mide, **sobre tu corrida**,
aprobación, mora observada y PD calibrada media de los aprobados en HO y OOT para una grilla de
cutoffs. Con esa tabla decides `CUTOFF_DECISION` (puede ser el mismo) y lo justificas. Los números
del resumen salen de aquí: no de tu Lab 2, que era otro modelo, ni de la demo de Austral.

In [ ]:
# 👇 TU CÓDIGO — insumos
CUTOFF_LAB2B = 540        # tu cutoff de partida: el que recomendaste en el Lab 2 · parte B; si no lo hiciste, deja 540 (referencia del curso)
CUTOFF_DECISION = ...        # el que propones para ESTE modelo (puede ser el mismo)
JUSTIFICACION_CUTOFF = """ESCRIBE AQUÍ: por qué este cutoff, con la aprobación y la mora que muestra la tabla."""

# ---- mecánica (viene escrita) ----
if "study" not in globals() or "comite" not in globals():
    print("⏭️ B5a pendiente: termina B2 y B3.")
elif not completo(CUTOFF_LAB2B, CUTOFF_DECISION, JUSTIFICACION_CUTOFF):
    print("⏭️ B5a pendiente: CUTOFF_LAB2B, CUTOFF_DECISION y su justificación.")
else:
    # Secciones del informe generado (la base de tu informe técnico): cada «## » del .qmd es un capítulo.
    texto_qmd = (SALIDA / "scorecard_report.qmd").read_text(encoding="utf-8")
    capitulos = [linea[3:].strip() for linea in texto_qmd.splitlines() if linea.startswith("## ")]
    secciones = [c for c in capitulos if not c.startswith("Anexo")]
    anexos = [c for c in capitulos if c.startswith("Anexo")]
    assert len(secciones) == 8 and len(anexos) == 3, (secciones, anexos)
    assert config_hash_actual in texto_qmd and data_hash_actual in texto_qmd
    print(f"informe generado (scorecard_report.qmd / .html): {len(secciones)} secciones + {len(anexos)} anexos, con tus hashes")
    for titulo in secciones + anexos:
        print(" ·", titulo)

    # Impacto del cutoff sobre TU corrida: grilla alrededor del cutoff del Lab 2 · parte B.
    grilla = sorted(set([int(CUTOFF_LAB2B) + d for d in (-40, -20, 0, 20, 40)] + [int(CUTOFF_DECISION)]))
    impacto = impacto_cutoff(study, grilla)
    print(f"\nIMPACTO DEL CUTOFF (aprobación, mora observada y PD calibrada media de los aprobados)")
    display(impacto.round(4))
    fila_decision = impacto[impacto["muestra"].eq("OOT") & impacto["cutoff"].eq(int(CUTOFF_DECISION))].iloc[0]
    APROBACION_OOT = float(fila_decision["aprobacion"])
    MALOS_OBS_OOT = float(fila_decision["malos_obs_aprobados"])
    PD_CAL_OOT = float(fila_decision["pd_cal_media_aprobados"])
    print(f"\nCon cutoff {int(CUTOFF_DECISION)} en OOT: aprueba {APROBACION_OOT:.1%} · mora observada entre "
          f"aprobados {MALOS_OBS_OOT:.2%} · PD calibrada media {PD_CAL_OOT:.2%}")
    print("Justificación:", JUSTIFICACION_CUTOFF)
    print("\n✅ B5a — informe listado y cutoff decidido sobre tu corrida")


**B5b — gatillos y resumen ejecutivo.** Tres gatillos de revalidación: cada uno con una
métrica de `VALORES_HOY`, un operador, un **umbral escrito por ti**, un responsable y una acción.
`disparado` lo calcula la celda contra el valor de hoy; nunca se escribe a mano. Un gatillo puede
no ser evaluable con esta corrida (`"metrica": None` y la condición en palabras), pero al menos dos
deben evaluarse con tus números.

Después, **tu** resumen ejecutivo: los cinco bloques del curso, en orden, con la decisión explícita
(el cutoff de B5a, su aprobación y su mora) y sin jerga sin traducir. Prueba de fuego: si al leerlo
no se sabe qué votar, es un abstract. La celda arma `resumen_ejecutivo_andes.md` con la identidad de
la corrida, la tabla de impacto, tus gatillos y tus cinco bloques.

**Invalidante de la tarea (B5 en 0):** resumen o informe que no corresponden al `run_id` entregado.

In [ ]:
# 👇 TU CÓDIGO — insumos
GATILLOS = [
    {"gatillo": ..., "metrica": ..., "operador": ..., "umbral": ..., "responsable": ..., "accion": ...},
    {"gatillo": ..., "metrica": ..., "operador": ..., "umbral": ..., "responsable": ..., "accion": ...},
    {"gatillo": ..., "metrica": ..., "operador": ..., "umbral": ..., "responsable": ..., "accion": ...},
]   # metrica: una clave de VALORES_HOY (o None + "condicion" en palabras) · operador: "<", "<=", ">", ">="

RESUMEN = {
    "1. Qué se pide aprobar": """ESCRIBE AQUÍ: una frase con versión, cutoff y vigencia.""",
    "2. Qué gana Financiera Andes": """ESCRIBE AQUÍ: aprobación y mora de TU corrida, en lenguaje de negocio.""",
    "3. Qué se sabe que no funciona": """ESCRIBE AQUÍ: dos o tres limitaciones traducidas (la calibración va aquí).""",
    "4. Qué se va a vigilar y con qué gatillo": """ESCRIBE AQUÍ: frecuencia y los gatillos con su dueño.""",
    "5. Qué se pide de vuelta": """ESCRIBE AQUÍ: la decisión pendiente, los recursos o el responsable a designar.""",
}

# ---- mecánica (viene escrita) ----
if "APROBACION_OOT" not in globals() or "model_card" not in globals():
    print("⏭️ B5b pendiente: termina B4 y B5a.")
else:
    metricas_hoy = comite["metricas"]
    hl = comite["calibracion"][comite["calibracion"]["test"].eq("hosmer_lemeshow")].set_index("partition")
    psi_tabla = comite["psi"].set_index("comparison")["value"]
    VALORES_HOY = {
        "gini_dev": float(metricas_hoy.loc["DEV", "gini"]),
        "gini_ho": float(metricas_hoy.loc["HO", "gini"]),
        "gini_oot": float(metricas_hoy.loc["OOT", "gini"]),
        "caida_gini_rel": float(1 - metricas_hoy.loc["OOT", "gini"] / metricas_hoy.loc["DEV", "gini"]),
        "ks_oot": float(metricas_hoy.loc["OOT", "ks"]),
        "psi_score_dev_ho": float(psi_tabla["dev_vs_holdout"]),
        "psi_score_dev_oot": float(psi_tabla["dev_vs_oot"]),
        "csi_max_dev_oot": float(comite["csi"]["value"].max()),
        "p_hl_ho": float(hl.loc["holdout", "p_value"]),
        "p_hl_oot": float(hl.loc["oot", "p_value"]),
        "brecha_calibracion_oot": float(hl.loc["oot", "observed_dr"] - hl.loc["oot", "expected_pd"]),
        "mora_aprobados_oot": MALOS_OBS_OOT,
    }
    print("VALORES DE HOY (los que un gatillo puede vigilar):")
    for clave, valor in VALORES_HOY.items():
        print(f" · {clave:24s} {valor:.4f}")

    bloques_ok = isinstance(RESUMEN, dict) and len(RESUMEN) == 5 and completo(*RESUMEN.values(), minimo=80)
    gatillos_ok = (isinstance(GATILLOS, list) and len(GATILLOS) == 3
                   and all(completo(g.get("gatillo", ...), g.get("responsable", ...), g.get("accion", ...), minimo=3)
                           for g in GATILLOS)
                   and all((g.get("metrica") is None and completo(g.get("condicion", ...), minimo=10))
                           or (g.get("metrica") in VALORES_HOY and g.get("operador") in OPERADORES
                               and isinstance(g.get("umbral"), (int, float)))
                           for g in GATILLOS)
                   and sum(g.get("metrica") is not None for g in GATILLOS) >= 2)
    if not gatillos_ok:
        print("\n⏭️ B5b pendiente: tres gatillos completos (métrica de VALORES_HOY + operador + umbral, "
              "o condición en palabras con metrica=None), cada uno con responsable y acción.")
    elif not bloques_ok:
        print("\n⏭️ B5b pendiente: los cinco bloques del resumen, cada uno escrito (≥ 80 caracteres).")
    else:
        tabla_gatillos = evaluar_gatillos(GATILLOS, VALORES_HOY)
        assert tabla_gatillos["responsable"].nunique() >= 2, "quien ejecuta no puede ser el único que decide: ≥ 2 responsables"
        texto_resumen = "\n".join(RESUMEN.values())
        assert str(int(CUTOFF_DECISION)) in texto_resumen, "el resumen debe nombrar tu cutoff"

        encabezado = (
            f"# Resumen ejecutivo — Financiera Andes · scorecard de admisión\n\n"
            f"**Preparado por:** {ID_ALUMNO} · **Corrida:** `run_id={run_id}` · "
            f"`config_hash={config_hash_actual}` · `data_hash={data_hash_actual}`\n\n"
            f"**Decisión medida sobre esta corrida (OOT):** cutoff **{int(CUTOFF_DECISION)}** → "
            f"aprobación {APROBACION_OOT:.1%} · mora observada de los aprobados {MALOS_OBS_OOT:.2%} · "
            f"PD calibrada media {PD_CAL_OOT:.2%}\n\n"
        )
        cuerpo = "".join(f"## {titulo}\n\n{texto.strip()}\n\n" for titulo, texto in RESUMEN.items())
        tabla_md = tabla_markdown(tabla_gatillos.reset_index())
        (SALIDA / "resumen_ejecutivo_andes.md").write_text(
            encabezado + cuerpo + "## Gatillos de revalidación\n\n" + tabla_md + "\n", encoding="utf-8")
        (SALIDA / "gatillos_andes.md").write_text(tabla_md + "\n", encoding="utf-8")
        assert run_id in (SALIDA / "resumen_ejecutivo_andes.md").read_text(encoding="utf-8")

        print("\nGATILLOS · estado calculado contra el valor de hoy")
        display(tabla_gatillos)
        print("\nRESUMEN EJECUTIVO")
        display(Markdown(encabezado + cuerpo))
        print(f"✅ B5b — {len(GATILLOS)} gatillos ({(tabla_gatillos['disparado'] == 'SÍ').sum()} disparado(s)) y "
              "resumen con decisión, ligados a la corrida")


---
## Exporta tu expediente

Esta celda empaqueta `salida_lab3b/` en `salida_lab3b_<tu id>.zip` y lo descarga desde Colab. Es
lo que adjuntas junto al link a este notebook y a los tres documentos. La matriz personal no va:
tu `data_hash` la identifica y la corrección la regenera con tu `ID_ALUMNO`.

In [ ]:
# ---- Exportar el expediente (NO modificar) ----
ESPERADOS = ["scorecard_report.html", "scorecard_report.qmd", "audit_trail_andes.jsonl",
             "lineage_andes.json", "model_card_andes.md", "model_card_andes.json",
             "resumen_ejecutivo_andes.md", "gatillos_andes.md"]
if not SALIDA.exists():
    print("⏭️ Todavía no hay expediente: corre B1 y B2.")
else:
    presentes = [a for a in ESPERADOS if (SALIDA / a).exists()]
    faltan = [a for a in ESPERADOS if a not in presentes]
    id_archivo = re.sub(r"[^a-z0-9]+", "_", ID_ALUMNO.strip().lower()).strip("_")
    zip_base = Path(f"salida_lab3b_{id_archivo}")
    ruta_zip = Path(shutil.make_archive(str(zip_base), "zip", root_dir=SALIDA))
    print(f"{ruta_zip.name}: {ruta_zip.stat().st_size / 1024:.0f} KiB")
    for archivo in presentes:
        print(" ·", archivo)
    if faltan or "tabla_gatillos" not in globals():
        print("⏭️ expediente incompleto — faltan:", ", ".join(faltan) or "completar B5b en esta sesión")
    else:
        print("✅ expediente completo: los ocho archivos de esta sesión están en el zip")
    try:
        from google.colab import files
        files.download(str(ruta_zip))
    except ImportError:
        print("(fuera de Colab: el zip quedó en el directorio de trabajo)")

---
# Opción 2 · Parte B a mano (tareas 7–12)

Si elegiste la opción 1, salta hasta el **Cierre y entrega**: estas celdas quedan en ⏭️.

## Puente · de la parte A a la parte B

La parte B empaqueta para producción **el mismo modelo** que acabas de validar: no hay nada
nuevo que reconstruir. Solo falta fijar **tu cutoff**, la decisión de política que tomaste en
el Lab 2, porque la corrida de producción lo necesita para decidir.

> Si en el Lab 2 recomendaste un cutoff distinto de 540, **cámbialo en la celda siguiente**; si no
> hiciste el Lab 2, deja 540 (el de referencia del curso). Lo que se evalúa es la coherencia con lo
> que argumentes, no el número en sí.

In [ ]:
# ---- Lo que la parte B necesita y la parte A no usaba ----
import copy
import json
import platform

# ---- Tu cutoff del Lab 2 · parte B (ajústalo al que TÚ recomendaste) ----
CUTOFF = 540    # ← si recomendaste otro en el Lab 2, cámbialo acá
print(f"cutoff de tu política: {CUTOFF} · "
      f"aprobación en OOT: {(SCORE_CAL['OOT'] >= CUTOFF).mean():.1%}")

---
## Las seis tareas de la opción 2

El modelo está probado. Ahora tiene que **existir fuera de este notebook**: congelado en un
artefacto, con contrato de datos, trazabilidad, ficha y gatillos escritos. Es **integración, no
analítica nueva**: todo lo que necesitas está en la clase 6 y en `demo_c6_bases_banco_austral.ipynb`.

---
## Tarea 7 — Congela tu modelo en un artefacto (3 puntos)

En producción **no existe DEV**. Tu `binear(x, ref=dev[v])` recalcula los
cortes mirando la muestra de desarrollo: correcto en el notebook, imposible en
el servidor de scoring.

Congelar es convertir esa partición en **datos**: cortes explícitos, WoE,
coeficientes, escalado, δ y master scale, todo en un JSON que viaja solo.

**7a.** Completa `ajustar_bins` y `aplicar_bins`, y arma tu `artefacto`.

In [ ]:
def es_numerica(s):
    """Numérica de verdad: el `StringDtype` de pandas NO lo es."""
    return pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

def clave_discreta(v):
    """0 y 0.0 son EL MISMO bin: sin esto, un NaN que promueva int64 a float64
    en producción cambia las claves y neutraliza la variable en silencio."""
    if pd.isna(v):
        return "MISSING"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return str(int(v)) if float(v).is_integer() else repr(float(v))
    return str(v)

def woe_normalizado(mapa, spec):
    """Reescribe las claves del WoE con la convención del bin congelado.

    En una variable discreta de tipo float, `binear` etiqueta "0.0" y el bin
    congelado "0". Es la MISMA partición con otro nombre — pero si el mapa
    queda a medio camino, la variable se neutraliza en silencio.
    """
    if spec["tipo"] != "discreto":
        return {str(b): float(w) for b, w in mapa.items()}
    salida = {}
    for b, w in mapa.items():
        if str(b) == "MISSING":
            salida["MISSING"] = float(w)
            continue
        try:
            salida[clave_discreta(float(b))] = float(w)
        except (TypeError, ValueError):
            salida[str(b)] = float(w)
    return salida

def misma_particion(a, b):
    """¿Dos etiquetados agrupan los mismos casos, aunque se llamen distinto?"""
    ca = pd.factorize(pd.Series(a).astype(str))[0]
    cb = pd.factorize(pd.Series(b).astype(str))[0]
    t = pd.crosstab(ca, cb).astype(bool)
    return bool(t.sum(axis=1).max() == 1 and t.sum(axis=0).max() == 1)

In [ ]:
def ajustar_bins(base, bins=5, umbral_moda=0.35):
    """Devuelve la MISMA partición que `binear(x, ref=base)`, pero como datos.

    Debe replicar la lógica de `binear`: categórica → sin cortes; pocos valores
    distintos → discreta; si no, cuantiles sobre el resto (separando la moda si
    concentra más de `umbral_moda`). Los ±inf se guardan como None (JSON válido).
    """
    base = pd.Series(base).reset_index(drop=True)
    if not es_numerica(base):
        return {"tipo": "categorico"}
    if base.nunique(dropna=True) <= bins:
        return {"tipo": "discreto",
                "valores": sorted({clave_discreta(v) for v in base.dropna()})}
    moda = base.mode().iloc[0]
    frac = float((base == moda).mean())
    # 👉 TU CÓDIGO: calcula `cortes` igual que `binear` (cuantiles sobre el
    #    resto si la moda concentra, sobre toda la base si no)
    resto = ...
    cortes = ...
    if len(cortes) < 3:
        return {"tipo": "discreto",
                "valores": sorted({clave_discreta(v) for v in base.dropna()})}
    cortes = [None] + [float(c) for c in cortes[1:-1]] + [None]
    spec = {"tipo": "numerico", "cortes": cortes, "moda_aparte": frac > umbral_moda}
    if spec["moda_aparte"]:
        spec["moda"] = int(moda) if float(moda).is_integer() else float(moda)
        spec["etiqueta_moda"] = f"= {moda:.4g}"
    return spec


def aplicar_bins(x, spec):
    """Asigna el bin con la spec congelada. PRESERVA el índice (la identidad)."""
    x = pd.Series(x)
    if spec["tipo"] == "categorico":
        return x.astype(str).where(x.notna(), "MISSING")
    if spec["tipo"] == "discreto":
        return pd.Series([clave_discreta(v) for v in x], index=x.index)
    interiores = [float(c) for c in spec["cortes"][1:-1]]
    # 👉 TU CÓDIGO: aplica pd.cut con [-inf, *interiores, +inf], conserva el
    #    índice, separa la moda si corresponde y marca los NaN como "MISSING"
    et = ...
    return ...

In [ ]:
# ---- Verificación automática (NO modificar): congelar no puede cambiar nada ----
try:
    _chk_7a = aplicar_bins(dev[ELEGIDAS[0]], ajustar_bins(dev[ELEGIDAS[0]]))
except Exception:
    _chk_7a = None
if _chk_7a is None:
    print("⏭️ Ítem 7a aún sin resolver — completa `ajustar_bins`/`aplicar_bins` antes de entregar")
else:
    comparaciones, renombradas = {}, set()
    for v in ELEGIDAS:
        spec = ajustar_bins(dev[v])
        for m, sub in [("DEV", dev), ("HO", ho), ("OOT", oot), ("TTD", ttd)]:
            cong = aplicar_bins(sub[v], spec)
            desa = binear(sub[v], ref=dev[v])[0]
            if (cong.values == desa.values).all():
                comparaciones[(v, m)] = True
            else:
                # una variable discreta de tipo float se etiqueta "0.0" en desarrollo
                # y "0" congelada: MISMA partición, otro nombre. Lo que no se admite
                # es que los casos caigan en grupos distintos.
                comparaciones[(v, m)] = misma_particion(cong, desa)
                if comparaciones[(v, m)]:
                    renombradas.add(v)
    n_ok = sum(comparaciones.values())
    assert n_ok == len(comparaciones), (
        f"el binning congelado cambia la PARTICIÓN en "
        f"{[k for k, v in comparaciones.items() if not v][:5]}")
    print(f"✅ 7a — misma partición en {n_ok} comparaciones "
          f"({len(ELEGIDAS)} variables × 4 muestras), sin usar DEV en el camino"
          + (f" · {len(renombradas)} discreta(s) con etiqueta normalizada: "
             f"{sorted(renombradas)}" if renombradas else ""))

**7b.** Arma el artefacto con TODO lo que producción necesita.

In [ ]:
def contrato_datos(dev, variables, tolerancia_missing=3.0):
    """Qué puede llegar: tipo, rango observado en DEV y missing tolerado."""
    clausulas = {}
    for v in variables:
        s = dev[v]
        tope = min(1.0, float(s.isna().mean()) * tolerancia_missing + 0.01)
        if not es_numerica(s):
            clausulas[v] = {"tipo": "categorico",
                            "categorias": sorted(s.dropna().astype(str).unique().tolist()),
                            "pct_missing_dev": float(s.isna().mean()),
                            "pct_missing_max": tope}
        else:
            clausulas[v] = {"tipo": "numerico",
                            "minimo": float(np.nanmin(s.values)),
                            "maximo": float(np.nanmax(s.values)),
                            "pct_missing_dev": float(s.isna().mean()),
                            "pct_missing_max": tope}
    return clausulas

# 👉 TU CÓDIGO: completa las claves que faltan. Regla de oro: si algo no está
#    en el artefacto, producción no lo tiene.
artefacto = {
    "modelo_id": "andes-scorecard-consumo",
    "cartera": "Crédito de consumo — Financiera Andes",
    "version": "1.0.0",
    "fecha_construccion": "2026-09-10",
    "autor_id": ID_ALUMNO,
    "variables": list(ELEGIDAS),
    "binning": ...,          # {variable: spec congelada}
    "woe": ...,              # {variable: {bin: woe}} — usa `woe_normalizado`
    "coeficientes": ...,     # {"const": ..., variable: ...}
    "escalado": {"pdo": 20, "score_base": 600, "odds_base": 50,
                 "factor": float(factor), "offset": float(offset)},
    "calibracion": ...,      # tendencia central, delta y el método
    "master_scale": {"cortes": CORTES_BANDA, "etiquetas": ETIQUETAS},
    "politica": {"cutoff": CUTOFF, "regla": "aprobar si score >= cutoff"},
    "contrato_datos": contrato_datos(dev, ELEGIDAS),
    "woe_faltante": 0.0,
}

In [ ]:
# ---- Verificación automática (NO modificar): el artefacto es JSON estándar ----
if artefacto is ... or any(v is ... for v in artefacto.values()):
    print("⏭️ Ítem 7b aún sin resolver — completa los ... de tu `artefacto` antes de entregar")
else:
    texto = json.dumps(artefacto, ensure_ascii=False, sort_keys=True, allow_nan=False)
    recargado = json.loads(texto)
    assert set(recargado) >= {"binning", "woe", "coeficientes", "escalado",
                              "calibracion", "master_scale", "politica",
                              "contrato_datos"}, "faltan claves en el artefacto"
    assert len(recargado["woe"]) == len(ELEGIDAS), "el mapa de WoE no cubre las variables"
    assert abs(recargado["calibracion"]["delta"] - delta) < 1e-12, "el δ no es el tuyo"
    print(f"✅ 7b — artefacto JSON válido: {len(texto):,} caracteres · "
          f"{sum(len(m) for m in artefacto['woe'].values())} bins con WoE · "
          f"sin NaN ni Infinity")

**7c.** La **prueba de paridad**: el artefacto solo tiene que producir el
mismo número que tu notebook. No «parecido»: el mismo.

Fíjate en dos detalles que parecen menores y no lo son: el **índice viaja con
el score** (si no, un merge posterior le puede asignar el score a otro
cliente), y los casos que caen en un bin que DEV nunca vio **salen marcados**
(un WoE neutro no es una estimación del riesgo: es declarar «no sé»).

In [ ]:
from scipy.special import expit, logit

def score_desde_art(p, art):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    return art["escalado"]["offset"] - art["escalado"]["factor"] * logit(p)

def puntuar(df, art):
    """PD → PD calibrada → score → banda → decisión, SOLO con el artefacto."""
    lp = np.full(len(df), art["coeficientes"]["const"], dtype=float)
    sin_mapa = np.zeros(len(df), dtype=int)
    for v in art["variables"]:
        # 👉 TU CÓDIGO: binea con la spec congelada, mapea el WoE, cuenta los
        #    que no encontraron mapa y acumula coeficiente × WoE en `lp`
        ...
    # 👉 TU CÓDIGO: PD cruda, PD calibrada con el δ, score y banda
    pd_cruda = ...
    pd_c = ...
    sc = ...
    salida = pd.DataFrame({"pd_calibrada": pd_c, "score": sc,
                           "banda": pd.cut(sc, [-np.inf] + art["master_scale"]["cortes"]
                                           + [np.inf],
                                           labels=art["master_scale"]["etiquetas"],
                                           right=False),
                           "bins_sin_mapa": sin_mapa,
                           "apto_automatico": sin_mapa == 0}, index=df.index)
    salida["decision"] = np.where(
        ~salida["apto_automatico"], "revisar",
        np.where(salida["score"] >= art["politica"]["cutoff"], "aprobar", "rechazar"))
    return salida

if artefacto is ... or any(v is ... for v in artefacto.values()):
    paridad = ...
    print("⏭️ La paridad espera tu artefacto del ítem 7b (y tu `puntuar` de arriba)")
else:
    paridad = pd.DataFrame([
        {"muestra": m,
         "n": len(MUESTRAS[m]),
         "max_dif_score": float(np.abs(puntuar(MUESTRAS[m], artefacto)["score"].values
                                       - SCORE_CAL[m]).max())}
        for m in ("DEV", "HO", "OOT", "TTD")]).set_index("muestra")
    print(paridad.to_string())

In [ ]:
# ---- Verificación automática (NO modificar): paridad e identidad ----
if "paridad" not in globals() or paridad is ...:
    print("⏭️ Ítem 7c aún sin resolver — necesita `puntuar` y tu artefacto (7b)")
else:
    assert paridad["max_dif_score"].max() < 1e-9, (
        f"la paridad no es exacta: máx {paridad['max_dif_score'].max():.2e}. "
        "Revisa el orden WoE → logit → δ → PDO.")
    sub = ttd.iloc[[3, 40, 200]]
    s_sub = puntuar(sub, artefacto)
    assert list(s_sub.index) == list(sub.index), "el scoring perdió la identidad del caso"
    uno = puntuar(ttd.iloc[[7]], artefacto)["score"].iloc[0]
    diez = puntuar(ttd.iloc[:10], artefacto)["score"].iloc[7]
    assert uno == diez, "el score de un caso depende del lote: hay estado escondido"
    salida_ttd = puntuar(ttd, artefacto)
    print(f"✅ 7c — paridad exacta (máx {paridad['max_dif_score'].max():.1e}), índice "
          f"preservado y score independiente del lote · aprobación en TTD: "
          f"{(salida_ttd['decision'] == 'aprobar').mean():.1%}")

**✍️ Tu justificación (7d):** ¿por qué la prueba de paridad tiene que dar
**exacta** y no «con una diferencia chica»? Si en tu caso diera 0,3 puntos de
score de diferencia, ¿qué buscarías primero, y por qué NO desplegarías igual?

`[tu respuesta acá]`

---
## Tarea 8 — El contrato de datos (3 puntos)

El modelo no sabe distinguir un dato bueno de uno malo: puntúa lo que le den.

Dos severidades, y **la fija la magnitud, no la regla**: 🔴 bloquea la corrida,
🟡 puntúa y deja constancia. Un 3% de valores fuera del rango de DEV es deriva
de población; un 100% es que alguien cambió la unidad.

**8a.** Completa el validador.

In [ ]:
TOLERANCIA_FUERA_RANGO = 0.01   # 1% fuera de rango: avisa
FUERA_RANGO_BLOQUEANTE = 0.10   # 10%: bloquea (eso es un cambio de unidad)
MISSING_BLOQUEANTE = 0.25       # 1 de cada 4 casos sin dato: el lote no sirve
FACTOR_BLOQUEO = 4.0

def validar_contrato(df, contrato):
    """Devuelve la tabla de hallazgos del lote contra el contrato."""
    hallazgos = []
    def anotar(v, sev, regla, detalle, valor, n_obs=None):
        hallazgos.append({"variable": v, "severidad": sev, "regla": regla,
                          "detalle": detalle, "valor": valor, "n_observado": n_obs})
    for v, c in contrato.items():
        if v not in df.columns:
            anotar(v, "🔴", "columna_faltante", "la columna no viene en el lote", None)
            continue
        s = df[v]
        num = es_numerica(s)
        if num != (c["tipo"] == "numerico"):
            anotar(v, "🔴", "tipo_incompatible",
                   f"esperado {c['tipo']}, llegó {'numerico' if num else 'categorico'}",
                   str(s.dtype))
            continue
        # 👉 TU CÓDIGO (missing): calcula el % de missing. Si supera el tope de
        #    la variable, anota 🟡; si además supera FACTOR_BLOQUEO× el tope o
        #    MISSING_BLOQUEANTE, anota 🔴.
        ...
        obs = s.dropna()
        if c["tipo"] == "numerico" and len(obs):
            # 👉 TU CÓDIGO (rango): % de los OBSERVADOS fuera de [minimo, maximo].
            #    Contar los NaN como "dentro" diluiría el indicador.
            #    > TOLERANCIA_FUERA_RANGO → 🟡 · > FUERA_RANGO_BLOQUEANTE → 🔴
            ...
        elif c["tipo"] == "categorico":
            # 👉 TU CÓDIGO (categorías): las que no estaban en DEV.
            ...
    return pd.DataFrame(hallazgos, columns=["variable", "severidad", "regla",
                                            "detalle", "valor", "n_observado"])

hallazgos_ttd = validar_contrato(ttd, artefacto["contrato_datos"])
print("Tu bandeja TTD real:")
print(hallazgos_ttd.to_string(index=False) if len(hallazgos_ttd) else "  (sin hallazgos)")

**8b.** Ahora el **lote roto de laboratorio**: los tres desastres clásicos
de un feed de producción. Ninguno lanza una excepción de Python — multiplicar
por mil es aritmética perfectamente válida y los NaN pandas los maneja sin
quejarse. Sin contrato, el lote se puntúa entero.

In [ ]:
# ---- El lote roto (NO modificar): tu feed de un martes cualquiera ----
# Los tres desastres se montan sobre variables APTAS de TU modelo: la unidad
# cambiada exige una numérica; el feed caído, una con tope de missing < 25%
# (si no, un 50% de NaN no bloquearía). Con tus ELEGIDAS del Lab 2 esto queda
# garantizado por selección, no por suerte.
if artefacto is ... or any(v is ... for v in artefacto.values()):
    lote_roto = None
    print("⏭️ Ítem 8 en espera — el lote roto se arma con tu artefacto del ítem 7b")
else:
    _con = artefacto["contrato_datos"]
    _numericas = [v for v in ELEGIDAS if _con[v]["tipo"] == "numerico"]
    assert _numericas, "tu modelo no tiene variables numéricas: caso fuera del diseño del lab"
    v0 = _numericas[0]
    v1 = next((v for v in ELEGIDAS if v != v0 and _con[v]["pct_missing_max"] < 0.25),
              None) or next(v for v in ELEGIDAS if v != v0)
    v2 = next(v for v in ELEGIDAS if v not in (v0, v1))
    lote_roto = ttd.copy()
    lote_roto[v0] = lote_roto[v0] * 1000                              # cambió la unidad
    lote_roto.loc[lote_roto.index[:len(lote_roto) // 2], v1] = np.nan # feed a la mitad
    lote_roto = lote_roto.drop(columns=[v2])                          # columna que no llegó
    print(f"lote roto: {len(lote_roto):,} filas · {v0} ×1000 · {v1} con 50% NaN · "
          f"{v2} ausente")

In [ ]:
# ---- Verificación automática (NO modificar): el contrato caza los tres ----
if lote_roto is None or validar_contrato(lote_roto.head(2), artefacto["contrato_datos"]) is None:
    print("⏭️ Ítem 8 aún sin resolver — necesita tu artefacto (7b) y `validar_contrato` (8a)")
else:
    h_roto = validar_contrato(lote_roto, artefacto["contrato_datos"])
    reglas = set(h_roto["regla"])
    if len(reglas) <= 1:
        print(f"⏭️ Ítem 8a a medio camino — tu validar_contrato caza {sorted(reglas)} "
              "de los tres desastres: sigue con las reglas que faltan")
    else:
        assert {"columna_faltante", "fuera_de_rango", "missing_excesivo"} <= reglas, (
            f"tu contrato no caza los tres desastres. Encontró: {sorted(reglas)}")
        rojas = h_roto[h_roto["severidad"] == "🔴"]
        assert {"fuera_de_rango", "missing_excesivo"} <= set(rojas["regla"]), (
            "la unidad cambiada (100% fuera de rango) y el feed caído (50% missing) "
            "tienen que BLOQUEAR, no solo avisar")
        assert (hallazgos_ttd["severidad"] != "🔴").all(), (
            "tu bandeja TTD real no debería tener bloqueantes: revisa los topes")
        print(h_roto.to_string(index=False))
        print(f"\n✅ 8 — contrato: {len(rojas)} 🔴 y "
              f"{(h_roto['severidad'] == '🟡').sum()} 🟡 en el lote roto · "
              f"tu bandeja real: {len(hallazgos_ttd)} hallazgo(s), ninguno bloqueante")

**✍️ Tu justificación (8c):** mira los hallazgos de tu **bandeja real** (no
del lote roto). Si hay alguno: ¿qué te está diciendo sobre tu cartera, y cómo
se conecta con el PSI/CSI de la parte A? Si no hay ninguno: ¿significa que tu
población no se movió, o que tus topes son demasiado anchos? Argumenta con
números.

`[tu respuesta acá]`

---
## Tarea 9 — La corrida reproducible, con audit trail (2 puntos)

Un audit trail responde tres preguntas: **qué se ejecutó, sobre qué datos y en
qué orden**. Cada evento sella al anterior con su hash.

**9a.** Completa el registro y córrelo de punta a punta.

In [ ]:
def hash_texto(t):
    return hashlib.sha256(t.encode("utf-8")).hexdigest()

def hash_json(obj):
    # sort_keys=True es lo que hace el hash comparable entre corridas
    return hash_texto(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str))

def hash_df(df):
    esquema = [{"pos": i, "nombre": str(c), "dtype": str(df[c].dtype)}
               for i, c in enumerate(df.columns)]
    h = hashlib.sha256()
    h.update(json.dumps(esquema, sort_keys=True, ensure_ascii=False).encode("utf-8"))
    h.update(pd.util.hash_pandas_object(df, index=True).values.tobytes())
    return h.hexdigest()

def snapshot_entorno():
    import scipy, sklearn, statsmodels
    return {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "statsmodels": statsmodels.__version__}

In [ ]:
class RegistroAuditoria:
    """Trail con hashes encadenados + verificación estricta."""

    def __init__(self, corrida_id):
        self.corrida_id = corrida_id
        self.eventos = []
        self._ts = 0            # reloj lógico: determinista

    def registrar(self, paso, evento, **payload):
        # 👉 TU CÓDIGO: arma el cuerpo del evento con n, corrida_id, t, paso,
        #    evento, payload y `hash_previo` (el hash del último evento, o 64
        #    ceros si es el primero). Después calcula su propio hash y guárdalo.
        ...
        return cuerpo

    def sello(self):
        """(hash terminal, nº de eventos): el ancla que se archiva FUERA del log."""
        if not self.eventos:
            return None, 0
        return self.eventos[-1]["hash"], len(self.eventos)

    def verificar_cadena(self, sello=None,
                         cierres=("corrida_terminada", "corrida_abortada")):
        """Devuelve (ok, motivo). Exige numeración, corrida única y cierre."""
        if not self.eventos:
            return False, "trail vacío: una corrida sin eventos no es evidencia"
        previo = "0" * 64
        for k, e in enumerate(self.eventos, start=1):
            if e["n"] != k:
                return False, f"numeración rota en la posición {k}"
            if e["corrida_id"] != self.corrida_id:
                return False, f"el evento {k} pertenece a otra corrida"
            cuerpo = {c: v for c, v in e.items() if c != "hash"}
            if cuerpo["hash_previo"] != previo:
                return False, f"el evento {k} no encadena con el anterior"
            if hash_json(cuerpo) != e["hash"]:
                return False, f"el contenido del evento {k} fue alterado"
            previo = e["hash"]
        if self.eventos[-1]["evento"] not in cierres:
            return False, "el trail no termina en un evento de cierre (fue truncado)"
        if sello is not None:
            h, n = sello
            if len(self.eventos) != n or self.eventos[-1]["hash"] != h:
                return False, "no coincide con el sello externo"
        return True, None

    def tabla(self):
        return pd.DataFrame([{"n": e["n"], "paso": e["paso"], "evento": e["evento"],
                              "hash": e["hash"][:12]} for e in self.eventos])

In [ ]:
# ---- La corrida completa (NO modificar): cada eslabón deja su hash ----
CRUDAS = {"clientes": clientes, "solicitudes": solicitudes,
          "comportamiento": comportamiento, "bureau": bureau}

def corrida_produccion(art, lote, crudas, registro=None):
    reg = registro or RegistroAuditoria(f"{art['modelo_id']}-{art['fecha_construccion']}")
    reg.registrar("inicio", "corrida_iniciada", modelo=art["modelo_id"],
                  version=art["version"], entorno=snapshot_entorno())
    reg.registrar("datos", "fuentes_crudas_leidas",
                  tablas={n: hash_df(t) for n, t in crudas.items()})
    presentes = [c for c in art["variables"] if c in lote.columns]
    reg.registrar("datos", "lote_seleccionado", muestra="TTD", n=int(len(lote)),
                  n_variables_presentes=len(presentes),
                  hash_lote=hash_df(lote[presentes]))
    h = validar_contrato(lote, art["contrato_datos"])
    bloqueantes = int((h["severidad"] == "🔴").sum())
    reg.registrar("contrato", "contrato_validado", n_hallazgos=int(len(h)),
                  bloqueantes=bloqueantes, reglas=sorted(h["regla"].unique().tolist()))
    if bloqueantes:
        reg.registrar("contrato", "corrida_abortada", motivo="hallazgos bloqueantes",
                      detalle=h[h["severidad"] == "🔴"][["variable", "regla"]]
                      .to_dict("records"))
        raise ValueError(f"contrato incumplido: {bloqueantes} hallazgo(s) 🔴")
    reg.registrar("modelo", "artefacto_cargado", hash_artefacto=hash_json(art),
                  n_variables=len(art["variables"]), delta=art["calibracion"]["delta"])
    s = puntuar(lote, art)
    reg.registrar("scoring", "lote_puntuado", n=int(len(s)),
                  score_medio=float(s["score"].mean()),
                  tasa_aprobacion=float((s["decision"] == "aprobar").mean()),
                  n_a_revision=int((s["decision"] == "revisar").sum()),
                  hash_salida=hash_df(s.astype({"banda": "string"})))
    reg.registrar("fin", "corrida_terminada", estado="ok")
    return s, reg

try:
    _reg_chk = RegistroAuditoria("chk"); _reg_chk.registrar("chk", "chk")
    _reg_listo = bool(_reg_chk.eventos)
except Exception:
    _reg_listo = False
if (artefacto is ... or any(v is ... for v in artefacto.values()) or not _reg_listo
        or validar_contrato(ttd.head(2), artefacto["contrato_datos"]) is None):
    salida = registro = SELLO = None
    print("⏭️ La corrida espera tu artefacto (7b), validar_contrato (8a) y el "
          "RegistroAuditoria (9) — nada de lo que falta bloquea las celdas siguientes")
else:
    salida, registro = corrida_produccion(artefacto, ttd, CRUDAS)
    SELLO = registro.sello()
    print(registro.tabla().to_string(index=False))
    print(f"\nsello: {SELLO[0][:16]}… · {SELLO[1]} eventos")

In [ ]:
# ---- Verificación automática (NO modificar): la cadena y el sello ----
if "registro" not in globals() or registro is None:
    print("⏭️ Ítem 9 aún sin resolver — necesita la corrida de producción (artefacto 7b + registro 9)")
else:
    ok, motivo = registro.verificar_cadena(sello=SELLO)
    assert ok, f"tu cadena nace rota: {motivo}"

    import copy
    # el "arreglito": editar sin resellar → la cadena lo caza
    torpe = copy.deepcopy(registro)
    n_obj = next(e["n"] for e in torpe.eventos if e["evento"] == "lote_puntuado")
    torpe.eventos[n_obj - 1]["payload"]["tasa_aprobacion"] = 0.99
    ok_t, motivo_t = torpe.verificar_cadena()
    assert not ok_t, "tu verificación no detecta una edición sin resellado"

    # el prolijo: editar Y recalcular la cola → solo lo caza el sello externo
    prolijo = copy.deepcopy(registro)
    prolijo.eventos[n_obj - 1]["payload"]["tasa_aprobacion"] = 0.99
    previo = "0" * 64
    for e in prolijo.eventos:
        e["hash_previo"] = previo
        e["hash"] = hash_json({k: v for k, v in e.items() if k != "hash"})
        previo = e["hash"]
    assert prolijo.verificar_cadena()[0], (
        "una cola recalculada DEBE pasar la cadena sola: por eso existe el sello")
    assert not prolijo.verificar_cadena(sello=SELLO)[0], (
        "el sello externo tiene que cazar la reescritura completa")

    # el contrato aborta y el aborto queda registrado
    try:
        corrida_produccion(artefacto, lote_roto, CRUDAS, RegistroAuditoria("intento-roto"))
        raise AssertionError("la corrida debió abortar con el lote roto")
    except ValueError:
        pass
    print(f"✅ 9 — trail de {len(registro.eventos)} eventos: caza la edición torpe "
          f"(«{motivo_t}»), la reescritura prolija solo la caza el sello externo, "
          f"y el lote roto aborta la corrida")

**✍️ Tu justificación (9b):** el ataque «prolijo» pasa la verificación de la
cadena. En una frase: **¿qué NO puede prometer** un audit trail encadenado sin
ancla externa? Y en dos más: ¿dónde guardarías el sello en Financiera Andes,
y por qué ahí?

`[tu respuesta acá]`

---
## Tarea 10 — Tu model card (3 puntos)

Una a dos páginas: **qué hace y para quién**, **qué tan bien lo hace y sobre
qué evidencia**, **qué NO se puede concluir de él**.

La regla que la hace útil: **se genera DESDE la corrida**, no se escribe aparte
de memoria. Un documento que no corresponde a los números del notebook es
invalidante.

In [ ]:
if artefacto is ... or any(v is ... for v in artefacto.values()) or registro is None:
    card = ...
    print("⏭️ Tu card se arma cuando tengas el artefacto (7b) y la corrida (9)")
else:
    # 👉 TU CÓDIGO: completa tu ficha con TUS números (los de este notebook y los
    #    de tu parte A). Las limitaciones son lo que más pesa: específicas, con
    #    número y con la consecuencia práctica al lado.
    card = {
        "identidad": {
            "modelo": artefacto["modelo_id"], "version": artefacto["version"],
            "tipo": "Scorecard logístico de admisión (PD a 12 meses)",
            "cartera": artefacto["cartera"],
            "fecha_construccion": artefacto["fecha_construccion"],
            "responsable_desarrollo": ID_ALUMNO,
            "hash_artefacto": hash_json(artefacto),
            "corrida_id": registro.corrida_id,
            "sello_trail": f"{SELLO[0]} ({SELLO[1]} eventos)"},
        "proposito": {
            "uso_previsto": ...,
            "usuarios": ...,
            "usos_no_previstos": [...]},        # al menos 4 — la sección que más se olvida
        "datos": {
            "poblacion": ...,
            "definicion_target": ...,
            "muestras": {m: int(len(MUESTRAS[m])) for m in ("DEV", "HO", "OOT", "TTD")},
            "tendencia_central": float(TC),
            "variables_finales": artefacto["variables"]},
        "desempeno": {
            "gini": ...,                        # ya lo calculaste arriba: reutiliza la variable
            "ks": ...,                          # idem — `metricas` sigue viva en memoria
            "psi_score_ttd": ...,               # idem — `psi_ttd`
            "backtesting_global_oot_p": ...,    # idem — `p_global`
            "tablero": ...},                    # idem — `tablero["estado"].value_counts()`
        "limitaciones": [...],                  # al menos 5, con números
        "gobierno": {
            "estado": "en validación independiente",
            "proxima_revalidacion": "2027-09-10",
            "frecuencia_monitoreo": ...,
            "tablero_vive_en": "el expediente del modelo"},
        "entorno": snapshot_entorno(),
    }

In [ ]:
# ---- Verificación automática (NO modificar): la ficha está completa ----
if ("card" not in globals() or card is ...
        or any(v is ... for v in card.get("desempeno", {}).values())
        or ... in card.get("limitaciones", [])):
    print("⏭️ Ítem 10 aún sin resolver — completa tu `card` (necesita además tu artefacto del 7b)")
else:
    assert set(card) == {"identidad", "proposito", "datos", "desempeno",
                         "limitaciones", "gobierno", "entorno"}, (
        f"faltan o sobran secciones: {sorted(card)}")
    assert len(card["proposito"]["usos_no_previstos"]) >= 4, (
        "declara al menos 4 usos NO previstos: es la sección que evita el problema "
        "más caro del gobierno de modelos")
    assert len(card["limitaciones"]) >= 5, "declara al menos 5 limitaciones"
    assert all(isinstance(l, str) and len(l) > 40 for l in card["limitaciones"]), (
        "«el modelo tiene supuestos» no es una limitación: cada una debe decir qué "
        "no se puede concluir y qué consecuencia práctica tiene")
    assert card["identidad"]["sello_trail"].startswith(SELLO[0]), (
        "la ficha debe sellar el trail de SU corrida")
    print(f"✅ 10 — model card con 7 secciones · "
          f"{len(card['proposito']['usos_no_previstos'])} usos no previstos · "
          f"{len(card['limitaciones'])} limitaciones declaradas")
    _punteros = sum(str(v).startswith("ver la parte A") for v in card["desempeno"].values())
    if _punteros:
        print(f"ℹ️ {_punteros} campo(s) de desempeño dicen «ver la parte A»: la pauta "
              "exige NÚMEROS de tu corrida — completa la parte A antes de entregar")

In [ ]:
# ---- Tu expediente en disco (NO modificar) ----
if "card" not in globals() or card is ...:
    print("⏭️ El expediente se escribe cuando tu card (ítem 10) esté completa")
else:
    def card_markdown(card):
        L = [f"# Model Card — {card['identidad']['modelo']} "
             f"v{card['identidad']['version']}", ""]
        titulos = {"identidad": "1. Identidad", "proposito": "2. Propósito y uso previsto",
                   "datos": "3. Datos", "desempeno": "4. Desempeño",
                   "limitaciones": "5. Limitaciones", "gobierno": "6. Gobierno",
                   "entorno": "7. Entorno de construcción"}
        for k, t in titulos.items():
            L += [f"## {t}", ""]
            v = card[k]
            if isinstance(v, list):
                L += [f"- {x}" for x in v]
            else:
                for k2, v2 in v.items():
                    etiqueta = k2.replace("_", " ").capitalize()
                    if isinstance(v2, dict):
                        L.append(f"- **{etiqueta}**:")
                        L += [f"  - {a}: `{b}`" for a, b in v2.items()]
                    elif isinstance(v2, list):
                        L.append(f"- **{etiqueta}**:")
                        L += [f"  - {x}" for x in v2]
                    else:
                        L.append(f"- **{etiqueta}**: {v2}")
            L.append("")
        return "\n".join(L)

    with open("model_card_andes.md", "w", encoding="utf-8") as f:
        f.write(card_markdown(card))
    with open("modelo_andes.json", "w", encoding="utf-8") as f:
        json.dump(artefacto, f, indent=1, ensure_ascii=False)
    with open("sello_andes.json", "w", encoding="utf-8") as f:
        json.dump({"hash_terminal": SELLO[0], "n_eventos": SELLO[1],
                   "corrida_id": registro.corrida_id, "autor": ID_ALUMNO}, f, indent=1)
    print("expediente escrito: model_card_andes.md · modelo_andes.json · sello_andes.json")
    print("\n(descárgalos desde el panel de archivos de Colab y adjúntalos a tu entrega)")

---
## Tarea 11 — Gatillos de revalidación (2 puntos)

El umbral se escribe **antes** de mirar el dato. Después de mirarlo, cualquier
umbral que elijas ya está contaminado por lo que viste — y todos en la sala lo
saben.

Un gatillo bien escrito tiene cinco partes: **condición** medible, **valor de
hoy**, **quién decide**, **qué acción** dispara y **en cuánto tiempo**.

In [ ]:
# 👉 TU CÓDIGO: completa la tabla con los valores de TU modelo. La columna
#    `disparado` NO se escribe a mano: se calcula desde la condición.
#    Si te falta el insumo (historial trimestral, estado del servicio), pon
#    "no evaluable" — no es lo mismo saber que no está disparado que no saberlo.
gini_oot = float(2 * roc_auc_score(oot["malo"], PD_CAL["OOT"]) - 1)
gini_dev = float(2 * roc_auc_score(dev["malo"], PD_CAL["DEV"]) - 1)
caida_rel = 1 - gini_oot / gini_dev

gatillos = pd.DataFrame([
    {"gatillo": "Vigilancia reforzada",
     "condicion": "cualquier indicador del tablero en 🟡 o 🔴",
     "valor_hoy": ...,          # de tu tablero de la parte A
     "disparado": ...,
     "responsable": "Modelador",
     "accion": "duplicar frecuencia de medición por 2 meses"},
    {"gatillo": "Recalibración del δ",
     "condicion": "binomial global 🟡 dos trimestres seguidos, o residuo de "
                  "nivel > 1 pto sostenido",
     "valor_hoy": ...,
     "disparado": ...,
     "responsable": "Jefe de Modelos",
     "accion": "re-anclar δ a la TC actualizada, con acta"},
    {"gatillo": "Re-desarrollo",
     "condicion": "caída relativa de Gini > 30%, o KS < 0,20, o PSI del score > 0,25",
     "valor_hoy": ...,          # ojo: son TRES ramas, evalúalas todas
     "disparado": ...,
     "responsable": "Comité de Riesgo",
     "accion": "abrir proyecto de re-desarrollo"},
    {"gatillo": "Contingencia",
     "condicion": "el modelo no puede puntuar (contrato roto) o cae el servicio",
     "valor_hoy": ...,
     "disparado": ...,
     "responsable": "TI / Producción",
     "accion": "política vigente de knock-outs como respaldo, con aviso al comité"},
]).set_index("gatillo")
print(gatillos.to_string())

In [ ]:
# ---- Verificación automática (NO modificar): los gatillos son controles ----
if gatillos is ... or (hasattr(gatillos, "columns") and "disparado" in gatillos.columns
                       and gatillos["disparado"].map(lambda x: x is ...).any()):
    print("⏭️ Ítem 11 aún sin resolver — completa tu tabla `gatillos` antes de entregar")
else:
    assert len(gatillos) >= 4, "faltan gatillos"
    assert set(gatillos.columns) >= {"condicion", "valor_hoy", "disparado",
                                     "responsable", "accion"}, "faltan columnas"
    assert gatillos["disparado"].isin(["SÍ", "no", "no evaluable"]).all(), (
        "`disparado` solo admite SÍ / no / no evaluable")
    assert gatillos["valor_hoy"].map(lambda s: isinstance(s, str) and len(s) > 8).all(), (
        "cada gatillo necesita su valor de HOY, no un placeholder")
    assert gatillos["responsable"].nunique() >= 2, (
        "si todos los gatillos son del mismo responsable, no hay separación de "
        "roles (quien ejecuta no puede ser el único que decide)")
    print(f"✅ 11 — {len(gatillos)} gatillos con condición, valor de hoy, dueño y "
          f"acción · {(gatillos['disparado'] == 'SÍ').sum()} disparado(s) · "
          f"{(gatillos['disparado'] == 'no evaluable').sum()} sin insumo para evaluar")

**✍️ Tu justificación (11b) — el diagnóstico (vale 1 de los 2 puntos):**
con tu tablero de la parte A y estos gatillos en la mano, escribe el párrafo que
llevarías al comité de Financiera Andes:

1. **Qué está pasando** con tu modelo (el patrón, no el semáforo suelto).
2. **Qué acción es proporcionada** y qué gatillo la respalda.
3. **Qué NO harías** y por qué.
4. **Quién firma** esa decisión.

`[tu respuesta acá]`

---
## Tarea 12 — El resumen ejecutivo de una página (2 puntos)

Para el comité, y responde una sola pregunta: **¿qué firmo y qué me pasa si lo
firmo?** Cinco bloques, en este orden.

Sin números concretos no es ejecutivo: es una carta de intenciones. Y si
alguien lo lee y no sabe qué tiene que decidir, no es un resumen ejecutivo —
es un abstract.

Nada de jerga sin traducir: «el Gini cayó pero el IC contiene el cero» se dice
«la capacidad de ordenar se mantiene; la baja que se ve cabe dentro del margen
de error de la muestra».

**✍️ Tu resumen ejecutivo (12a).** Escríbelo acá, en una página:

### 1. Qué se pide aprobar
`[una frase, con versión y vigencia]`

### 2. Qué gana Financiera Andes
`[con números de negocio, no métricas técnicas]`

### 3. Qué se sabe que no funciona
`[dos o tres limitaciones, traducidas]`

### 4. Qué se va a vigilar y con qué gatillo
`[frecuencia + los gatillos duros con su dueño]`

### 5. Qué se pide de vuelta
`[la decisión pendiente, los recursos o el responsable a designar]`

> 📄 Este mismo texto va en el documento aparte
> (`PLANTILLA_resumen_ejecutivo.md`, en el anexo 2 de este notebook), que se entrega junto
> al informe técnico.

---
## Cierre y entrega

Checklist antes de entregar:

- [ ] `Reiniciar y ejecutar todo` corre sin errores y **los 13 ✅ aparecen**: 6 en la parte A (ítems 1–6) y 7 en la opción de parte B que elegiste (opción 1: B1, B2, B3, B4, B5a, B5b y «expediente completo» · opción 2: 7a, 7b, 7c, 8, 9, 10 y 11). La otra opción queda en ⏭️
- [ ] Las seis lecturas de la **parte A** (1b a 6b) están escritas y `ACCIONES` cubre tus indicadores en 🔴
- [ ] Opción 1: regla propia con cero filas fuera de rango · lectura del trail · juicio sobre las diferencias con tu parte A · ≥ 3 limitaciones propias · cutoff decidido sobre la tabla de impacto · tres gatillos con `disparado` calculado · resumen con decisión
- [ ] Opción 2: artefacto JSON estándar con **paridad exacta** · el contrato bloquea el lote roto y no tu bandeja real · trail que verifica contra el sello · model card con ≥ 4 usos no previstos y ≥ 5 limitaciones con números · gatillos con `disparado` calculado · justificaciones 7d, 8c, 9b, 11b y 12a
- [ ] Los tres documentos de la E3 citan los identificadores de **esta** corrida (`run_id`/`config_hash`/`data_hash`, o hash del artefacto y sello del trail) y sus números son los de estas celdas
- [ ] Descargaste la evidencia: `salida_lab3b_<tu id>.zip` (opción 1) o `modelo_andes.json` + `sello_andes.json` + `model_card_andes.md` (opción 2)
- [ ] Tu `ID_ALUMNO` es tu correo de inscripción (el mismo de los Labs 1 y 2, si los hiciste)

### La Entrega 3 completa

| Pieza | Opción 1 (nikodym) | Opción 2 (a mano) |
|---|---|---|
| Lab 3 ejecutado (partes A y B) | este notebook en Colab, corrido de arriba a abajo | igual |
| Informe técnico (PDF o MD) | `scorecard_report.qmd` completado, o `PLANTILLA_informe_tecnico.md` con las tablas del informe generado | `PLANTILLA_informe_tecnico.md` llenada |
| Model card | `model_card_andes.md` (B4) + desempeño de B3, usos no previstos y gobierno | `model_card_andes.md` (tarea 10) o la plantilla llenada |
| Resumen ejecutivo (1 página) | `resumen_ejecutivo_andes.md` (B5b) pulido en la plantilla | tu tarea 12 pulida en la plantilla |
| Evidencia de la corrida | `salida_lab3b_<tu id>.zip` | `modelo_andes.json` · `sello_andes.json` |

**Vence el domingo 27 de septiembre, 23:59.** Notas y feedback: viernes 2 de octubre.

> 🎓 Con esto tu proyecto está completo por construcción: seis clases, tres
> labs, y un expediente que otra persona podría auditar sin llamarte.

---
# Anexo 1 · Cómo se evalúa la Entrega 3 (la pauta)

Esta es la pauta completa de la Entrega 3, la misma con la que se corrige. Léela antes de empezar:
está escrita para que sepas exactamente contra qué se corrige.

### Un solo notebook, una pauta

**La Entrega 3 es una sola:** el Lab 3 (`lab_3_financiera_andes.ipynb`, un solo archivo) + los **tres
documentos**. El notebook trae la **parte A** (ítems 1–6, clase 5) y **dos opciones de parte B, a
elección**: **opción 1 con `nikodym`** (B1–B5) u **opción 2 a mano** (tareas 7–12). Haces **una
sola**; las dos valen 15 puntos y se corrigen con la misma pauta (si entregas ambas, se corrige la de
`nikodym`). Se evalúan los mismos objetos que siempre: notebook reproducible, informe técnico, model
card y resumen ejecutivo. Las equivalencias de más abajo dicen qué evidencia corresponde en cada opción.

**Opción 1 · con nikodym** (parte B del Lab 3, tareas B1–B5)

1. Ejecuta el notebook con tu `ID_ALUMNO`: la mecánica viene escrita y la parte B reutiliza lo que ya
   calculaste en la parte A (tu muestra, tu TC, tu Gini del ítem 1, tus `ELEGIDAS`); tú pones la regla
   del contrato, el cutoff, y escribes las lecturas, las limitaciones, los gatillos y el resumen. El
   informe generado (`scorecard_report.qmd`, Markdown editable) es la **base**
   del informe técnico, no la entrega: completa sus bloques «POR COMPLETAR» (introducción, contexto,
   veredicto, conclusiones) y agrega lo que la corrida no sabe: política vigente, fugas descartadas,
   cutoff con su impacto (B5a), gatillos (B5b), limitaciones propias y usos no previstos.
2. **El informe documenta el modelo de tu corrida nikodym**, no el de tu Lab 2. Tu Lab 2 y tu Lab 3 ·
   parte A entran como **comparación** (B3), nunca mezclados como si fueran el mismo modelo: un informe
   con el scorecard de nikodym y el bootstrap del Lab 3A sin decir de qué modelo es cada cosa cae en el
   invalidante n.º 1.
3. Cita `run_id`, `config_hash` y `data_hash` de tu corrida en el informe, el model card y el resumen:
   son tu «artefacto y sello». El model card generado (B4) trae identidad, datos, supuestos, limitaciones
   y decisiones; **tú agregas** la tabla de desempeño (Gini/KS por muestra, PSI, Hosmer-Lemeshow — los de
   B3), los usos no previstos y el gobierno.

**Opción 2 · a mano / desde cero** (parte B del Lab 3, tareas 7–12)

1. Usa las plantillas de `04_proyecto/` para construir el informe, el model card y el resumen ejecutivo;
   el artefacto (`modelo_andes.json`), el sello (`sello_andes.json`) y el model card los genera tu tarea 10.
2. Usa como checklist las nueve piezas del expediente de la lámina 7 de la clase 6.
3. Haz que cada cifra y artefacto sea trazable al notebook ejecutado con tu `ID_ALUMNO`.

**Equivalencias entre caminos** (la rúbrica pide la misma evidencia; cambia el archivo donde vive):

| La pauta dice… | Camino manual | Camino nikodym |
|---|---|---|
| Artefacto y sello de tu corrida | `modelo_andes.json` (sha256) · `sello_andes.json` | `run_id` · `config_hash` · `data_hash` (en `lineage_andes.json`, la ficha y el informe) + `audit_trail_andes.jsonl` |
| Contrato de datos | tarea 8 (validador con dos severidades) | B1 (esquema de 114 columnas + tu regla de rango; `check_dataset` avisa, `run` bloquea) |
| Model card generado desde la corrida | tarea 10 → `model_card_andes.md` | B4 → `model_card_andes.md` (+ desempeño y usos no previstos que agregas tú) |
| Gatillos con `disparado` calculado | tarea 11 | B5b → `gatillos_andes.md` |
| Borrador del resumen ejecutivo | tarea 12 | B5b → `resumen_ejecutivo_andes.md` |
| Cutoff con su impacto | Lab 2 · parte B (mismo modelo) | B5a: tabla de aprobación y mora en HO/OOT **de tu corrida** (el Lab 2 era otro modelo) |
| Evidencia que se adjunta | `modelo_andes.json` · `sello_andes.json` | `salida_lab3b_<tu id>.zip` (informe, trail, lineage, model card, resumen, gatillos) |

**Qué necesitas para aprobar (E3 ≥ 6,0) y qué suma para la nota alta.** La fórmula no cambia
(§2). Con la parte A completa y correcta en lo mecánico, aprueba un proyecto que muestra, en
cualquier camino: **una corrida reproducible sobre tu muestra personal** (semilla = tu `ID_ALUMNO`,
identificadores citados), **validación fuera de desarrollo** (HO y OOT, nunca solo DEV), **un
diagnóstico honesto que incluya la calibración** (en Andes el nivel de PD suele rechazar: decirlo y
proponer la acción proporcionada vale más que esconderlo), **una recomendación con condiciones**,
**limitaciones propias con consecuencia práctica** y **coherencia** entre notebook, informe, model card
y resumen. La nota alta no exige pasos técnicos extra: exige **profundidad y trazabilidad** — el porqué
de cada decisión, limitaciones que descubriste tú, gatillos con dueño y umbral, un resumen con el que
un comité puede votar. **Opcional** (suma solo si está bien hecho, no resta si falta): intervalo
bootstrap o Hosmer-Lemeshow en el camino donde tu notebook no lo calcula, pérdida esperada en pesos con
las convenciones del Lab 2, informe en Word/PDF además del Markdown.

El invalidante común es el mismo: **un informe que no corresponde al notebook**. La E3 vence el
**domingo 27 de septiembre** —fecha máxima— y las notas finales están propuestas para el
**viernes 2 de octubre**.

---

### 1. Qué se entrega y cuándo

**Fecha límite: domingo 27 de septiembre, 23:59 (hora de Chile).**
Notas y feedback: **viernes 2 de octubre**.

El Lab 3 se publica el **miércoles 16 de septiembre**, completo y en un solo archivo (parte A de
la clase 5 y parte B de la clase 6, por la opción que elijas).

> ⚠️ **Ojo con las fechas.** El plazo se corrió una semana con la reprogramación del curso
> (era el mié 16, después el mié 23). Ahora las **Fiestas Patrias (18–19) caen DENTRO del plazo**,
> no después: en la práctica pierdes dos días hábiles (18 y 19). La entrega es el **domingo 27**
> y el feedback el **viernes 2 de octubre**. Es la fecha máxima: no se corre más.

| Pieza | Formato | Opción 1 (nikodym) | Opción 2 (a mano) |
|---|---|---|---|
| **Lab 3** ejecutado (partes A y B) | link a Colab, celdas corridas en orden | `lab_3_financiera_andes.ipynb` con B1–B5 | el mismo notebook con las tareas 7–12 |
| **Informe técnico** | PDF o Markdown | `scorecard_report.qmd` completado (o la plantilla con las tablas del informe generado) | plantilla `PLANTILLA_informe_tecnico.md` |
| **Model card** | Markdown | `model_card_andes.md` (B4) + desempeño, usos no previstos y gobierno | `model_card_andes.md` (tarea 10) completado, o la plantilla llenada |
| **Resumen ejecutivo** | 1 página, PDF o Markdown | `resumen_ejecutivo_andes.md` (B5b) pulido en la plantilla | `PLANTILLA_resumen_ejecutivo.md` desde tu tarea 12 |
| **Evidencia de la corrida** | archivos adjuntos | `salida_lab3b_<tu id>.zip` (lo genera la última celda) | `modelo_andes.json` · `sello_andes.json` |

> Los notebooks se entregan **ejecutados**: celdas corridas en orden, con sus
> salidas visibles. Un notebook escrito pero no ejecutado se corrige solo hasta
> donde alcanzó a correr.

---

### 2. Cómo se calcula la nota

La E3 vale **50% del curso** y es la que define el certificado.

| Componente | Peso en E3 | Puntaje máximo |
|---|---|---|
| **Lab 3 · parte A** — validación y monitoreo | **40%** | 12 puntos |
| **Lab 3 · parte B + proyecto final** | **60%** | 35 puntos |

**Nota E3** = 0,4 × nota(parte A) + 0,6 × nota(parte B + proyecto)

Cada componente se lleva a escala 1–10 con la conversión estándar del curso:
`nota = 1 + 9 × (puntaje / máximo)`.

#### Desglose del componente «parte B + proyecto final» (35 puntos)

| Pieza | Puntos | Qué se evalúa |
|---|---|---|
| Parte B del notebook | **15** | Manual: artefacto y paridad · contrato de datos · corrida con audit trail · model card · gatillos · resumen ejecutivo. Nikodym: config y contrato (B1) · corrida y trail leído (B2) · objetos de comité comparados con tus labs (B3) · model card con limitaciones propias (B4) · cutoff, gatillos y resumen (B5) |
| Informe técnico | **10** | Las 7 secciones + anexos, con el razonamiento de cada decisión |
| Model card (documento) | **5** | Ficha completa, con usos no previstos y limitaciones específicas (desglose en §3) |
| Resumen ejecutivo | **5** | Una página que permita a un comité decidir (desglose en §3) |

> **Por qué el model card y el resumen aparecen dos veces.** Dentro del notebook (tareas
> 10 y 12 del camino manual; B4 y B5 del camino nikodym) se puntúa que **se generen desde tu
> corrida**: que salgan del artefacto o de la corrida, con sus identificadores, y no escritos a
> mano aparte. Como documento entregado se puntúa el **contenido final**: qué dicen y si sirven
> para decidir. Son dos cosas distintas y por eso suman por separado.

> El proyecto pesa más que la parte A porque es el entregable que define el certificado.
> La parte A es la de la clase 5; la parte B y los documentos cierran la entrega.

#### Condición de aprobación

- **E3 ≥ 6,0** para aprobar el proyecto y obtener el certificado.
- Si E3 < 6,0: **una reentrega** dentro de 7 días, con nota máxima **7,0**.
- **Nota final del curso** = 0,2 × E1 + 0,3 × E2 + 0,5 × E3.
- No hay requisito de asistencia: las clases quedan grabadas.

#### Atrasos

−0,5 por día, máximo 3 días. Después se considera no entregada (nota 1,0),
salvo causa justificada avisada antes del plazo. «Fecha máxima» significa que el plazo
no se corre más; el atraso se descuenta.

---

### 3. Qué se evalúa en el informe técnico (10 puntos)

| Sección | Puntos | Qué tiene que estar |
|---|---|---|
| 1–2 · Introducción y contexto | 1,0 | Qué decide el modelo, sobre qué población, contra qué política vigente |
| 3 · Metodología | 3,0 | Target y ventanas justificados · exclusiones · esquema muestral · fábrica de variables · fugas descartadas con argumento temporal · embudo de selección con números · scorecard · calibración |
| 4 · Resultados | 1,5 | Scorecard, master scale, cutoff con su análisis de impacto, swap-set. *Camino nikodym:* scorecard y deciles del informe generado + la tabla de impacto del cutoff de B5a; el swap-set no se exige (es un objeto del Lab 2, de otro modelo) |
| 5 · Validación formal | 2,0 | Discriminación con IC · estabilidad · backtesting · tablero con acción y dueño por indicador · **y el diagnóstico del patrón** (§5.6: qué está fallando y por qué, no solo qué indicador se puso rojo). *Camino nikodym:* Gini/KS, PSI/CSI y Hosmer-Lemeshow del informe generado, comparados con tu Lab 3A (B3); el IC bootstrap es opcional — si no lo calculas, decláralo como limitación; el tablero son tus gatillos de B5b con dueño y acción |
| 6 · Conclusiones y recomendación | 1,0 | Qué se pide aprobar, con qué condiciones, plan de implementación y gobierno. Incluye el **bloque «Resumen ejecutivo» del propio informe**: se puntúa acá, no aparte |
| 7 · Limitaciones y supuestos | 1,0 | Específicas, con números y con su consecuencia práctica |
| Anexos A–C | 0,5 | Lineage y reproducibilidad · tablas · parámetros completos |

**Lo que distingue un informe de 10 de uno de 6:** el de 10 explica **por qué**
cada decisión, no solo cuál. «Descarté `canal` por PSI 0,685» es un dato;
«descarté `canal` porque su PSI de 0,685 indica que la mezcla de canales cambió
entre desarrollo y la bandeja actual, y una variable así traslada al score un
cambio comercial que no es riesgo» es un argumento.

#### El model card como documento (5 puntos)

| Qué se mira | Puntos | Qué tiene que estar |
|---|---|---|
| Ficha completa y trazable | 1,5 | Las 7 secciones, con el hash de **tu** artefacto y el sello de **tu** trail (camino nikodym: `run_id`, `config_hash` y `data_hash` de **tu** corrida, los mismos de B2) |
| Desempeño con números | 1,5 | Gini, KS, PSI y backtesting de tus tres muestras — los mismos del notebook, no punteros (camino nikodym: la ficha generada no los trae; los agregas tú desde B3, con el Hosmer-Lemeshow como backtesting) |
| Usos NO previstos | 1,0 | Al menos 4, específicos y con la razón al lado |
| Limitaciones | 1,0 | Al menos 5, cada una con su número y su consecuencia práctica |

#### El resumen ejecutivo como documento (5 puntos)

| Qué se mira | Puntos | Qué tiene que estar |
|---|---|---|
| Los cinco bloques, en orden | 1,5 | Qué se pide · qué gana el negocio · qué no funciona · qué se vigila · qué se decide |
| Números de negocio | 1,5 | Aprobación y PD calibrada media de los aprobados — **de tu notebook**; la pérdida esperada en pesos suma si la calculas con las convenciones declaradas del Lab 2 (LGD 45%, EAD = monto). Nada de métricas de modelador sin traducir |
| Decisión pedida explícita | 1,0 | Al terminar de leerlo se sabe qué hay que firmar y bajo qué condiciones |
| Una página, sin jerga | 1,0 | Cabe en una plana y un gerente lo entiende sin glosario |

---

### 4. Dimensiones de rúbrica (las mismas del curso)

| Dimensión | Peso | Qué mira en la E3 |
|---|---|---|
| **A. Corrección técnica** | 40% | Métricas bien calculadas y sobre la muestra correcta · paridad del artefacto (manual) o identificadores de la corrida coherentes entre notebook y documentos (nikodym) · contrato bien implementado o bien declarado · tests con el p-valor adecuado |
| **B. Juicio y negocio** | 25% | Lee el **patrón**, no el semáforo suelto · acción proporcionada con su gatillo · limitaciones que el alumno descubrió, no las que copió |
| **C. Reproducibilidad** | 20% | Corre de punta a punta · semilla personal · artefacto que reproduce el score · lineage completo |
| **D. Comunicación** | 15% | Informe claro · resumen ejecutivo que permite decidir · tablas legibles |

---

### 5. Invalidantes (llevan la E3 a un máximo de 4,0)

1. **Cualquiera de los tres documentos (informe, model card o resumen ejecutivo)
   reporta números que no están en el notebook.** Es el error más caro y el más
   fácil de detectar: si el notebook dice Gini 0,631 y el documento dice 0,66
   porque quedó de una corrida anterior, el documento completo pierde
   credibilidad. Una sola regla para los tres.
2. **Métricas reportadas solo en DEV**, sin contraste en HO/OOT.
3. **El artefacto no reproduce el score del notebook** (paridad > 1e-9) y se
   entrega igual, sin declararlo. (Camino manual. En el camino nikodym el artefacto es la
   corrida misma: su equivalente es que `run_id`, `config_hash` y `data_hash` del notebook
   sean los que citan el informe, el model card y el resumen — si no coinciden, aplica el n.º 1.)
4. **Números de otra semilla**: el trabajo no corresponde al `ID_ALUMNO`
   declarado.
5. **Modelo ajustado sobre una matriz que no es la propia** (arrastrado desde
   entregas anteriores pese al feedback).

---

### 6. Integridad

- Cada alumno trabaja su subsample personal de Financiera Andes (semilla =
  `ID_ALUMNO`). Tus números son individuales por construcción.
- **Usa el mismo `ID_ALUMNO` de los tres labs.** Si cambia, tus resultados
  dejan de ser trazables entre entregas.
- La corrección se hace contra **rangos esperados por semilla**: el generador
  permite recomputar la verdad de cada subsample.
- Copia evidente (números de otra semilla, código idéntico con comentarios
  ajenos) → nota 1,0 en la entrega y aviso.

**Sobre las celdas de referencia:** si tu Lab 1 o tu Lab 2 quedó incompleto, el
Lab 3 trae celdas de referencia que te dejan avanzar. Usarlas **no
penaliza**, siempre que lo declares en la justificación 6b de tu notebook (y lo
menciones en el informe). Se declara y se sigue; no se
copia en silencio.

---

### 7. Checklist antes de enviar

- [ ] El notebook corre con `Reiniciar y ejecutar todo` sin errores.
- [ ] Las celdas de verificación muestran sus ✅: 6 en la parte A y 7 en la opción de parte B que hiciste (la otra queda en ⏭️).
- [ ] El `ID_ALUMNO` es el mismo en los tres labs.
- [ ] Todos los números del informe salen de los notebooks entregados.
- [ ] Las limitaciones son específicas, con números y con consecuencia práctica.
- [ ] El tablero tiene **acción y dueño** por indicador, no solo semáforos.
- [ ] El resumen ejecutivo permite saber **qué hay que decidir**.
- [ ] Adjuntaste la evidencia de tu corrida: `salida_lab3b_<tu id>.zip` (opción 1) o `modelo_andes.json` y `sello_andes.json` (opción 2).
- [ ] Los tres documentos citan los identificadores de **esa** corrida (hash y sello, o `run_id`/`config_hash`/`data_hash`) y describen **un solo modelo**.
- [ ] Borraste las notas en cita (`>`) de las plantillas.

---

> 🎓 Con esta entrega tu proyecto queda completo: seis clases, tres labs y un
> expediente que otra persona podría auditar sin llamarte. Eso — no el
> scorecard — es lo que te llevas del curso.

---
# Anexo 2 · Las tres plantillas del proyecto

Copia cada plantilla a tu documento (Word, Google Docs o Markdown), reemplaza lo que está `[entre
corchetes]` y borra las notas en cita (`>`) antes de entregar. Si hiciste la parte B con `nikodym`,
tu base es el informe generado (`scorecard_report.qmd`) y la model card de B4: las notas de cada
plantilla dicen qué completar en ese caso.

Las plantillas citan como insumo «Lab 1» y «Lab 2»: si no los hiciste, ese insumo está en las
**celdas de referencia** de este notebook (target y muestras, fábrica de variables, modelo y
scorecard, calibración) y en las clases 1 a 4. Basta con describir lo que esas celdas hacen con tus
números.

## Plantilla · Informe técnico

## Informe técnico de modelo — [NOMBRE DEL MODELO]

> **Cómo usar esta plantilla.** No es un formato del curso: es la estructura
> estándar de la industria — la misma que emiten las herramientas de reporte y
> la que espera un validador. Cada sección indica **de qué clase sale su
> contenido**, **qué debe tener** y **qué la invalida**.
>
> **Jerarquía de sanciones, para que no haya sorpresas:** lo que una nota de
> sección marca como «invalidante» invalida **esa sección** (su puntaje va a 0);
> los invalidantes de **la entrega completa** (tope de nota 4,0) son solo los
> cinco publicados en el anexo 1 §5.
>
> Reemplace todo lo que está `[entre corchetes]`. Borre las notas en cita (`>`)
> antes de entregar: son instrucciones, no contenido.
>
> **Regla madre: todo número de este informe tiene que salir de su notebook.**
> Si el notebook dice Gini 0,631 y acá dice 0,66 porque quedó de una corrida
> anterior, el documento completo pierde credibilidad — y es invalidante.
>
> **Este informe se ARMA, no se escribe de cero.** Todo el insumo ya existe en
> sus tres labs: las tablas están impresas en sus notebooks y varias secciones
> ya las redactó como justificaciones (✍️). Lo que este documento agrega es el
> **argumento** — el porqué de cada decisión — no un análisis nuevo. Use la
> tabla siguiente como índice de armado, y respete los topes: **más largo no es
> mejor** — un validador premia densidad. Los topes son máximos, no metas: un
> cuerpo de **~10–12 páginas** ya es un informe completo (los anexos van aparte
> y pueden ir por referencia).
>
> | Sección del informe | El insumo ya existe en… | Tope |
> |---|---|---|
> | Resumen ejecutivo (bloque) | su `PLANTILLA_resumen_ejecutivo` (borrador: tarea 12 del Lab 3) | ½ página |
> | 1–2 · Introducción y contexto | enunciados del Lab 1 + su tabla de cosechas (Lab 2 · ítem 10) | 1 página |
> | 3.1–3.4 · Target, ventanas, población, muestras | Lab 1 · ítems 2–5 y sus justificaciones | 1½ páginas |
> | 3.5–3.6 · Fábrica y fugas | Lab 1 · ítems 7–10 + la doctrina del ítem 12 | ½ página |
> | 3.7–3.9 · Selección, binning, modelo y scorecard | Lab 2 · parte A, con sus 1c · 3c · 5c | 1½ páginas |
> | 3.10 · Calibración | Lab 2 · parte B (ítems 9–13) | ½ página |
> | 4 · Resultados | Lab 2 · ítems 7, 13 y 14 + su recomendación 14b | 2 páginas |
> | 5 · Validación formal | Lab 3 · parte A (ítems 1–6), con sus 1b · 3b · 4b · 5b · 6b | 2½ páginas |
> | 6 · Conclusiones y recomendación | su 14b + 11b + tareas 7–9 y 11 (artefacto, contrato, corrida, gatillos) | 1½ páginas |
> | 7 · Limitaciones y supuestos | las limitaciones de su model card (tarea 10), con su consecuencia | 1 página |
>
> Adaptar sus propias justificaciones es exactamente lo esperado; lo que se
> evalúa es que el argumento sea suyo y calce con sus números.
>
> **Si hiciste la parte B con `nikodym`:** tu base es `scorecard_report.qmd`, el
> informe generado por tu corrida (Markdown editable; el `.html` es el mismo
> documento). Ya trae, **del modelo de tu corrida**, las secciones 3.7–3.10,
> 4.1–4.2, 5.1–5.4 y los anexos A–C. Completa sus bloques «POR COMPLETAR» con
> las secciones 1–2 y 3.1–3.6 de esta plantilla (salen de tu Lab 1, iguales en
> ambos caminos); agrega 4.3 con la tabla de impacto del cutoff de B5a (4.4, el
> swap-set, no se exige en este camino), 5.5–5.6 con tus gatillos y tu
> diagnóstico (B5b) y 6–7 con tu recomendación y tus limitaciones (B4). Cita
> `run_id`, `config_hash` y `data_hash`. **No mezcles** en las secciones 4–5
> números de tu Lab 2 o Lab 3A: son otro modelo y van solo como comparación (B3).

| Campo | Valor |
|---|---|
| Modelo | `[id-del-modelo]` |
| Versión | `[1.0.0]` |
| Cartera | Crédito de consumo — Financiera Andes |
| Autor | `[Su nombre]` · `ID_ALUMNO = [su id]` |
| Fecha | `[AAAA-MM-DD]` |
| Estado | En validación / Aprobado / En producción |
| Notebook de respaldo | `[link a Colab, ejecutado en orden]` |
| Hash del artefacto | `[sha256 del JSON del modelo]` — camino nikodym: `run_id` · `config_hash` · `data_hash` |

---

### Resumen ejecutivo

> **Media página, escrita al final y leída primero.** Para quien no va a leer
> el resto. Cinco frases bastan: qué se pide aprobar, qué gana la institución,
> qué se sabe que no funciona, qué se va a vigilar y con qué gatillo, y qué se
> pide de vuelta. Con números concretos: sin ellos es una carta de intenciones.
>
> El documento aparte de una página para el comité va en
> `PLANTILLA_resumen_ejecutivo.md` — este bloque es su versión condensada.

`[Escriba aquí]`

---

### 1. Introducción

> **De la clase 1.** Qué problema de negocio resuelve este modelo, qué decisión
> concreta toma, sobre qué población y con qué alcance. Y — no lo omita — qué
> queda **fuera** del alcance.

#### 1.1 Problema de negocio y decisión que se automatiza
`[¿Qué se decide con este score? ¿Quién lo usa? ¿Con qué frecuencia?]`

#### 1.2 Alcance y exclusiones
`[Producto, segmento, canal. Qué NO cubre.]`

#### 1.3 Modelo anterior o política vigente
`[¿Contra qué se compara? Si hoy se decide con knock-outs, descríbalos.]`

**Invalidante:** describir un modelo de admisión y luego evaluarlo como si
fuera de comportamiento (o viceversa).

---

### 2. Contexto del modelo y de la cartera

> **De la clase 1.** El lector tiene que poder juzgar si sus resultados son
> razonables para ESTA cartera, y para eso necesita conocerla.

#### 2.1 Descripción de la cartera
`[Tamaño, producto, montos y plazos típicos, perfil de clientes.]`

#### 2.2 Tasa de malos histórica y su evolución
`[Tabla de cosechas con tasa de malos. Comente estabilidad y maduración.]`

#### 2.3 Política de crédito vigente
`[Reglas duras actuales, tasa de aprobación, mora observada.]`

---

### 3. Metodología

> **De las clases 1 a 4.** El corazón del informe. Un validador tiene que poder
> **reproducir** su modelo leyendo esta sección — no adivinar qué hizo.

#### 3.1 Definición del target
`[Default = 90+ DPD en ventana de 12 meses desde el punto de observación.
Justifique la ventana con roll rates y curvas de maduración de cosechas.
Explique el tratamiento de indeterminados y por qué. Ojo: dias_mora es la mora
consolidada del CLIENTE — si su curva de maduración no parte de cero, explique
la mora heredada por la política de admisión (cursa con vigente ≤ 59 días) en
vez de reportarla como anomalía.]`

#### 3.2 Ventanas y punto de observación
`[Diagrama o descripción de la ventana de observación (hacia atrás) y de
desempeño (hacia adelante). Deje explícito que cada solicitud lleva SU propia
ventana.]`

#### 3.3 Exclusiones y población final
`[Fraude, antigüedad mínima, duplicados. Cuántos casos salen por cada regla.]`

#### 3.4 Esquema muestral
`[DEV / HO / OOT / TTD: criterio de partición, tamaños, tasa de malos de cada
una, y para qué existe cada muestra.]`

#### 3.5 Fábrica de variables
> **De la clase 2.**

`[Familias de variables, ventanas 3/6/12m, agregadores, ratios de negocio.
Cuántas candidatas quedaron. Anclaje temporal al punto de observación.]`

#### 3.6 Fugas de información detectadas y descartadas
`[Qué variables descartó por mirar el futuro y CON QUÉ ARGUMENTO TEMPORAL.]`

#### 3.7 Selección de variables
> **De las clases 2 y 3.**

`[Embudo completo con números: candidatas → estables por PSI → IV ≥ umbral →
correlación → VIF → stepwise. Una tabla del embudo y el argumento por cada
variable descartada que sea relevante.]`

#### 3.8 Binning y WoE
`[Método de binning, tratamiento de missing, monotonía. IV calculado SOLO
sobre DEV.]`

#### 3.9 Modelo y scorecard
> **De la clase 3.**

`[Especificación logística, coeficientes con signos y p-valores, escalado PDO
(factor, offset, odds base) y tabla de puntos.]`

#### 3.10 Calibración
> **De la clase 4.**

`[Tendencia central: cómo se ancló y con qué cosechas. δ aplicado. Curva de
calibración. Master scale con sus bandas.]`

**Invalidantes de esta sección:** target que mira hacia atrás o ventana mal
anclada; IV calculado sobre toda la muestra; particiones con contaminación
temporal; usar una variable con fuga sabiéndolo.

---

### 4. Resultados

> **De las clases 3 y 4.** Qué produce el modelo y qué decisión permite tomar.

#### 4.1 Scorecard final
`[Tabla de puntos por variable y bin.]`

#### 4.2 Distribución del score y master scale
`[Distribución por muestra, ocupación de bandas, PD por banda.]`

#### 4.3 Estrategia y cutoff propuesto
`[Tabla de cutoffs con aprobación, mora esperada y pérdida esperada.
Recomendación fundada, no solo la tabla.]`

#### 4.4 Swap-set contra la política vigente
`[Quién entra, quién sale, y qué pasa con la mora a igual aprobación.]`

#### 4.5 Reason codes
`[Ejemplo de 2–3 casos con sus principales razones de rechazo.]`

---

### 5. Validación formal

> **De la clase 5.** La evidencia. Cada afirmación con su número y su muestra.

#### 5.1 Discriminación
`[AUC / Gini / KS en DEV, HO y OOT. Intervalo de confianza por bootstrap
(camino nikodym: tu corrida no lo calcula; si no lo agregas, decláralo como
limitación). Lectura HONESTA de la caída en OOT: ¿es concluyente o cabe en el ruido?]`

#### 5.2 Tabla de deciles
`[Captura de malos y lift por decil en OOT — la tabla del ítem 2 de tu notebook. El KS por decil es opcional (tu notebook reporta el KS global del ítem 1).]`

#### 5.3 Estabilidad
`[PSI del score contra OOT y contra TTD. CSI de las variables finales.
Identifique la variable que más se movió y diga por qué.]`

#### 5.4 Backtesting de calibración
`[Test binomial global y por banda (ítem 5 de su notebook). Hosmer-Lemeshow es
opcional: se enseñó en la clase 5 con el caso Austral; si lo calcula, declare qué p-valor está
usando y por qué — si los esperados por grupo son pequeños, la aproximación
χ² exagera. Camino nikodym: el backtesting de TU corrida es el Hosmer-Lemeshow por
muestra que imprime B3 (el binomial de tu parte A es del otro modelo).]`

#### 5.5 Tablero de monitoreo
`[Los indicadores con su umbral, su valor de hoy, su semáforo, su frecuencia
y — sobre todo — su ACCIÓN y su dueño. Un indicador sin acción es un adorno.]`

#### 5.6 Diagnóstico
`[La parte que más pesa: lea el PATRÓN, no el semáforo suelto. ¿Qué está
pasando con este modelo? ¿Qué acción es proporcionada? ¿Qué NO haría, y por
qué? ¿Quién firma esa decisión?]`

**Invalidante:** reportar métricas solo en DEV.

---

### 6. Conclusiones y recomendación

> Qué se pide aprobar, con qué condiciones y por cuánto tiempo. Sea explícito:
> un informe que termina en «el modelo tiene un desempeño adecuado» no permite
> tomar ninguna decisión.

#### 6.1 Recomendación
`[Aprobar / aprobar con condiciones / no aprobar. Con cutoff y vigencia.]`

#### 6.2 Condiciones de aprobación
`[Qué tiene que cumplirse para usarlo, y qué se revisa antes de renovarlo.]`

#### 6.3 Plan de implementación
> **De la clase 6.**

`[Batch u online. Contrato de datos. Versionamiento. Qué se registra en cada
decisión. Plan de contingencia si el servicio cae o el contrato se rompe.]`

#### 6.4 Gobierno
`[Roles (quién desarrolla, quién valida, quién aprueba, quién opera),
frecuencia de monitoreo, gatillos de revalidación con su dueño y su plazo.]`

---

### 7. Limitaciones y supuestos

> **La sección que separa un informe profesional de un trabajo de curso.**
> Genéricas no sirven: «el modelo tiene supuestos» no es una limitación.
> Cada una debe decir **qué no se puede concluir** y **qué consecuencia
> práctica tiene**.

| # | Limitación | Consecuencia práctica |
|---|---|---|
| 1 | `[ej.: modelado sobre aprobados: no observa el X% rechazado]` | `[describe a quien la institución ya dejaba entrar]` |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

#### Supuestos declarados
`[LGD utilizada, EAD, horizonte, estabilidad del entorno macro, etc.]`

---

### Anexo A — Lineage y reproducibilidad

> **De la clase 6.** Cómo se demuestra que estos números salieron de estos
> datos con este modelo.

| Eslabón | Identificador / hash |
|---|---|
| Tablas crudas | `[sha256 de cada una]` |
| Matriz de modelación | `[sha256]` |
| Artefacto del modelo | `[sha256]` — nikodym: `config_hash` |
| Salida de scoring | `[sha256]` — nikodym: `data_hash` de la matriz personal |
| Corrida | `[corrida_id]` — nikodym: `run_id` |
| Sello del audit trail | `[hash terminal + nº de eventos]` — nikodym: nº de eventos y decisiones de `audit_trail_andes.jsonl` |

#### Entorno de ejecución
`[Python, numpy, pandas, scipy, scikit-learn, statsmodels y cualquier otra librería usada,
con su versión exacta — p. ej. `nikodym[scoring]==1.11.0` si trabajaste con la pista RiskLib.]`

#### Semillas
`[Semilla personal (ID_ALUMNO) y cualquier otra semilla usada.]`

---

### Anexo B — Tablas detalladas

> Si la tabla ya está impresa en su notebook entregado, **basta la referencia**
> (notebook y nº de ítem): pegue aquí solo las que ayuden a leer el informe sin
> abrir el notebook.

`[Embudo de selección completo · tabla WoE/IV de las variables finales ·
matriz de correlación · VIF · deciles por muestra · master scale completa —
pegadas o referenciadas a su notebook.]`

---

### Anexo C — Parámetros completos

`[El artefacto: variables, cortes de binning, mapa de WoE, coeficientes,
factor/offset, δ de calibración, cortes de la master scale y cutoff.
Pegue el JSON, o referencie el artefacto de su expediente (nombre de archivo +
sha256): es el mismo objeto, ya sellado por su corrida. Camino nikodym: el Anexo C
del informe generado ya trae los parámetros completos; basta referenciarlo junto a
`lineage_andes.json`.]`

---

> **Checklist antes de entregar**
>
> - [ ] Todos los números del informe salen del notebook entregado.
> - [ ] El notebook corre de punta a punta, en orden, sin errores.
> - [ ] Está usado el `ID_ALUMNO` de siempre (el mismo de los tres labs).
> - [ ] Las limitaciones son específicas y traen números.
> - [ ] El tablero tiene acción y dueño por indicador, no solo semáforos.
> - [ ] Se borraron todas las notas en cita de esta plantilla.


## Plantilla · Model card

## Model Card — [id-del-modelo] v[1.0.0]

> **Qué es.** Una a dos páginas que un gerente lee en tres minutos y que un
> validador usa como índice. Responde tres preguntas: **qué hace y para quién**,
> **qué tan bien lo hace y sobre qué evidencia**, y **qué NO se puede concluir
> de él**.
>
> **Regla:** se genera DESDE la corrida, no se escribe aparte de memoria. Y no
> puede contar una historia distinta a la del tablero: si el tablero tiene tres
> indicadores en rojo, la ficha los dice.
>
> **Punto de partida:** la tarea 10 de su Lab 3 ya genera esta ficha con sus
> números, su hash y su sello. Este documento se arma DESDE esa salida —
> complete lo que falta (usos no previstos, limitaciones con consecuencia,
> gobierno), no la reescriba de cero.
>
> **Camino nikodym:** la tarea B4 genera `model_card_andes.md` con identidad
> (`run_id`, `config_hash`, `data_hash`), datos, supuestos, limitaciones (las
> suyas + las automáticas) y decisiones del trail. Esa ficha **no trae** la tabla
> de desempeño ni los usos no previstos ni el gobierno: agréguelos aquí con los
> números de B3 y los gatillos de B5b. Los identificadores deben ser los de B2.

### 1. Identidad

- **Modelo**: `[id-del-modelo]`
- **Versión**: `[1.0.0]`
- **Tipo**: Scorecard logístico de admisión (PD a 12 meses)
- **Cartera**: Crédito de consumo — Financiera Andes
- **Fecha de construcción**: `[AAAA-MM-DD]`
- **Responsable de desarrollo**: `[su nombre]` · `ID_ALUMNO [id]`
- **Hash del artefacto**: `[sha256]` (camino nikodym: `config_hash`)
- **Corrida**: `[corrida_id]` · sello del trail `[hash terminal + nº eventos]`
  (camino nikodym: `run_id` · `data_hash` · nº de eventos y decisiones del trail)

### 2. Propósito y uso previsto

- **Uso previsto**: `[decisión de admisión de crédito de consumo con cutoff X]`
- **Usuarios**: `[motor de originación automático · mesa de excepciones]`
- **Usos NO previstos** (la sección que más se olvida y la que evita el
  problema más caro del gobierno de modelos — el modelo que alguien reutilizó
  para otra cosa «porque total, da una probabilidad»):
  - `[ej.: provisiones IFRS 9 — requiere PD lifetime, no a 12 meses]`
  - `[ej.: cobranza o cartera vigente — es un modelo de admisión]`
  - `[ej.: precios diferenciados — no fue validado para ese uso]`
  - `[ej.: segmentos fuera de consumo]`

### 3. Datos

- **Población**: `[descripción y período]`
- **Definición de target**: `[default = 90+ DPD en 12 meses desde el punto de
  observación]`
- **Muestras**: DEV `[n]` · HO `[n]` · OOT `[n]` · TTD `[n]`
- **Tasa de malos en DEV**: `[x,x%]` · **Tendencia central**: `[x,x%]`
- **Variables finales** (`[k]`): `[lista]`
- **Hash de la matriz de modelación**: `[sha256]`

### 4. Desempeño

| Métrica | DEV | HO | OOT |
|---|---|---|---|
| Gini | | | |
| KS | | | |

- **IC 95% del Gini en OOT** (bootstrap): `[lo; hi]` *(camino nikodym: opcional; si no lo calcula, escriba «no calculado» — una ficha honesta declara sus límites)*
- **PSI del score contra TTD**: `[x,xxx]` *(camino nikodym: DEV→OOT, la corrida no puntúa la TTD)*
- **CSI máximo contra TTD**: `[x,xxx]` (`[variable]`) *(camino nikodym: DEV→OOT)*
- **Backtesting binomial global (OOT)**: p = `[x,xx]` *(camino nikodym: Hosmer-Lemeshow OOT de B3, con su decisión)*
- **Hosmer-Lemeshow (OOT)** *(opcional en el camino manual — el Lab 3 no lo calcula; si lo agrega, salga de su corrida)*: `[estadístico y p-valor, declarando cuál usa]`
- **Tablero de monitoreo**: `[n]` 🟢 · `[n]` 🟡 · `[n]` 🔴 *(camino nikodym: sus gatillos de B5b y cuántos están disparados)*

### 5. Limitaciones

> Específicas, con números, y con la consecuencia práctica al lado. Un modelo
> sin limitaciones declaradas no es un modelo sin limitaciones: es un modelo
> sin examinar.

- `[limitación 1 — con su número]`
- `[limitación 2]`
- `[limitación 3]`
- `[limitación 4]`
- `[limitación 5]`

### 6. Gobierno

- **Estado**: `[en desarrollo / en validación independiente / aprobado]`
- **Próxima revalidación**: `[AAAA-MM-DD]`
- **Frecuencia de monitoreo**: `[mensual discriminación y estabilidad ·
  trimestral calibración]`
- **Gatillos de acción**: `[referencia a los gatillos de revalidación del informe (§6.4) y al tablero de monitoreo (§5.5)]`
- **Dónde vive el tablero**: `[el expediente del modelo]`
- **Responsable del monitoreo**: `[rol]` · **Quién aprueba cambios**: `[rol]`

### 7. Entorno de construcción

- **Python**: `[versión]`
- **Librerías**: `[numpy, pandas, scipy, scikit-learn, statsmodels con versiones — y nikodym==1.11.0 si es su camino]`
- **Semilla**: `[ID_ALUMNO]`
- **Reproducibilidad**: `[declare lo que NO puede probar: sin repositorio git,
  sin archivo de bloqueo de dependencias, etc. Una ficha honesta declara sus
  propios límites.]`


## Plantilla · Resumen ejecutivo

## Resumen ejecutivo — [id-del-modelo] v[1.0.0]

> **Una página. Para el comité.** Responde una sola pregunta: *¿qué firmo y qué
> me pasa si lo firmo?*
>
> Cinco bloques, en este orden. El comité lee de arriba hacia abajo y se
> detiene cuando ya sabe qué votar. **Sin números concretos no es ejecutivo: es
> una carta de intenciones.** Y si alguien lo lee y no sabe qué tiene que
> decidir, no es un resumen ejecutivo — es un abstract.
>
> Nada de jerga sin traducir. «El Gini cayó 10,8% en OOT pero el IC contiene el
> cero» se dice: «la capacidad de ordenar se mantiene; la baja que se ve cabe
> dentro del margen de error de la muestra».
>
> **Punto de partida:** la tarea 12 de su Lab 3 ya es el borrador de esta
> página, escrito con sus números. Aquí se pule para un comité — no se
> redacta de nuevo. (Camino nikodym: el borrador es `resumen_ejecutivo_andes.md`
> de B5b, que ya trae los cinco bloques, los identificadores de la corrida y la
> aprobación y mora de su cutoff.)

**Fecha:** `[AAAA-MM-DD]` · **Preparado por:** `[su nombre]`

---

### 1. Qué se pide aprobar

`[Una frase, con versión y vigencia. Ej.: «Uso del scorecard de admisión
v1.0.0 con cutoff 540 para crédito de consumo, por 12 meses.»]`

### 2. Qué gana la institución

`[El beneficio, con números de negocio — no con métricas técnicas. Aprobación,
PD calibrada media de los aprobados (su notebook la imprime; camino nikodym: la tabla de impacto de B5a, en OOT, con la mora observada al lado); pérdida esperada en pesos si la calcula con las convenciones declaradas del Lab 2 (LGD 45%, EAD = monto); comparación contra la política de referencia.]`

### 3. Qué se sabe que no funciona

`[Las limitaciones, traducidas a lenguaje de negocio. Dos o tres, las que
cambian cómo se usa el modelo. Ej.: «subestima el riesgo de solicitantes que
la política actual rechazaba: mora observada [X]% contra [Y]% previsto».
Usa TUS números: copiar los de la demo se corrige como copia.]`

### 4. Qué se va a vigilar y con qué gatillo

`[Frecuencia del tablero, y los dos o tres gatillos duros con su condición, su
acción y quién la firma. Ej.: «si el backtesting queda amarillo dos trimestres
seguidos, se recalibra con acta del comité».]`

### 5. Qué se pide de vuelta

`[La decisión de negocio pendiente, los recursos o el nombre del responsable
que hay que designar. Si no se pide nada, dígalo.]`
